In [3]:
# =============================================================================
# NOTEBOOK 13 — CELL 2
# FINAL 1976-CONDITION GENOMIC PREPARATION
# READ-ONLY GENOMIC SOURCE FORENSIC COMPARISON
# =============================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("NOTEBOOK 13 — CELL 2")
print("GENOMIC SOURCE FORENSIC COMPARISON")
print("=" * 80)

# =============================================================================
# 1. AUTHORITATIVE PROJECT ROOTS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

CXR_ROOT = (
    PROJECT_ROOT /
    "Step_3B_8_CXR_CoAtNet_Preparation"
)

FINAL_ROOT = (
    CXR_ROOT /
    "QC" /
    "Clean_PSPNet_CXR_Cohort" /
    "FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT /
    "MASTER_CONDITION_LEVEL_SPLIT"
)

# =============================================================================
# 2. GENOMIC SOURCE CANDIDATES
# =============================================================================

SOURCE_A = (
    PROJECT_ROOT /
    "Step_3B_6_55_Genomic_Normal_Candidate_Forensic_Review" /
    "03_Candidate_Conditions_Full_Genomic_Records.csv"
)

SOURCE_B = (
    PROJECT_ROOT /
    "TB_Portals_Genomics_March_2025.csv"
)

# =============================================================================
# 3. AUTHORITATIVE FROZEN SPLIT
# =============================================================================

FULL_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

TRAIN_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_TRAIN.csv"
)

VALIDATION_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_TEST.csv"
)

# =============================================================================
# 4. OUTPUT
# =============================================================================

OUTPUT_ROOT = (
    FINAL_ROOT /
    "Genomic_Preparation_13"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 5. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

paths = {

    "Final split":
        FULL_SPLIT_FILE,

    "Train split":
        TRAIN_SPLIT_FILE,

    "Validation split":
        VALIDATION_SPLIT_FILE,

    "Test split":
        TEST_SPLIT_FILE,

    "Source A":
        SOURCE_A,

    "Source B":
        SOURCE_B,

}

for name, path in paths.items():

    exists = path.exists()

    print(
        f"{name:<25}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

# =============================================================================
# 6. LOAD FROZEN SPLITS
# =============================================================================

full_df = pd.read_csv(
    FULL_SPLIT_FILE
)

train_df = pd.read_csv(
    TRAIN_SPLIT_FILE
)

validation_df = pd.read_csv(
    VALIDATION_SPLIT_FILE
)

test_df = pd.read_csv(
    TEST_SPLIT_FILE
)

required_split_columns = {
    "condition_id",
    "target_binary"
}

for name, df in {

    "full": full_df,
    "train": train_df,
    "validation": validation_df,
    "test": test_df,

}.items():

    missing = (
        required_split_columns
        - set(df.columns)
    )

    if missing:

        raise RuntimeError(
            f"{name} split missing columns: "
            f"{sorted(missing)}"
        )

    df["condition_id"] = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )

# =============================================================================
# 7. FINAL COHORT GATE
# =============================================================================

print("\nFINAL COHORT GATE")
print("-" * 80)

expected_sizes = {
    "full": 1976,
    "train": 1185,
    "validation": 395,
    "test": 396,
}

loaded_splits = {
    "full": full_df,
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for name, expected in expected_sizes.items():

    actual = len(
        loaded_splits[name]
    )

    print(
        f"{name:<12}: {actual}"
    )

    if actual != expected:

        raise RuntimeError(
            f"{name} split size mismatch."
        )

print(
    "Cohort sizes: PASS"
)

# =============================================================================
# 8. CONDITION UNIQUENESS
# =============================================================================

print("\nCONDITION UNIQUENESS")
print("-" * 80)

for name, df in loaded_splits.items():

    unique_count = (
        df["condition_id"]
        .nunique()
    )

    print(
        f"{name:<12}: "
        f"{unique_count}"
    )

    if unique_count != len(df):

        raise RuntimeError(
            f"Duplicate conditions in {name}."
        )

print(
    "Condition uniqueness: PASS"
)

# =============================================================================
# 9. SPLIT OVERLAP
# =============================================================================

train_ids = set(
    train_df["condition_id"]
)

validation_ids = set(
    validation_df["condition_id"]
)

test_ids = set(
    test_df["condition_id"]
)

print("\nSPLIT OVERLAP")
print("-" * 80)

train_val = (
    train_ids
    .intersection(validation_ids)
)

train_test = (
    train_ids
    .intersection(test_ids)
)

val_test = (
    validation_ids
    .intersection(test_ids)
)

print(
    "Train ∩ Validation:",
    len(train_val)
)

print(
    "Train ∩ Test:",
    len(train_test)
)

print(
    "Validation ∩ Test:",
    len(val_test)
)

if train_val or train_test or val_test:

    raise RuntimeError(
        "Condition overlap detected."
    )

print(
    "Split integrity: PASS"
)

# =============================================================================
# 10. TARGET DISTRIBUTION
# =============================================================================

print("\nTARGET DISTRIBUTION")
print("-" * 80)

expected_targets = {

    "full": {
        0: 704,
        1: 1272,
    },

    "train": {
        0: 422,
        1: 763,
    },

    "validation": {
        0: 141,
        1: 254,
    },

    "test": {
        0: 141,
        1: 255,
    },

}

for name, df in loaded_splits.items():

    counts = (
        df["target_binary"]
        .value_counts()
        .to_dict()
    )

    ds = int(
        counts.get(0, 0)
    )

    dr = int(
        counts.get(1, 0)
    )

    print(
        f"{name:<12}: "
        f"DS={ds} DR={dr}"
    )

    if {
        0: ds,
        1: dr,
    } != expected_targets[name]:

        raise RuntimeError(
            f"Target distribution mismatch "
            f"in {name}."
        )

print(
    "Target distribution: PASS"
)

# =============================================================================
# 11. FROZEN COHORT IDS
# =============================================================================

frozen_ids = set(
    full_df["condition_id"]
)

if len(frozen_ids) != 1976:

    raise RuntimeError(
        "Frozen cohort does not contain "
        "1976 unique conditions."
    )

# =============================================================================
# 12. LOAD BOTH GENOMIC CANDIDATES
# =============================================================================

print("\nGENOMIC SOURCE LOADING")
print("-" * 80)

df_a = pd.read_csv(
    SOURCE_A,
    low_memory=False
)

df_b = pd.read_csv(
    SOURCE_B,
    low_memory=False
)

print(
    "Source A rows:",
    len(df_a)
)

print(
    "Source B rows:",
    len(df_b)
)

# =============================================================================
# 13. SCHEMA INSPECTION
# =============================================================================

print("\nSCHEMA INSPECTION")
print("-" * 80)

required_genomic_columns = [

    "condition_id",
    "main_lineage",
    "sub_lineage",
    "drug_resistance_type",
    "gene_snp_mutations",

]

for source_name, df in {

    "Source A": df_a,
    "Source B": df_b,

}.items():

    missing = [
        column
        for column in required_genomic_columns
        if column not in df.columns
    ]

    print(
        f"{source_name} required schema:",
        "PASS" if not missing else "FAIL"
    )

    if missing:

        print(
            "Missing:",
            missing
        )

# =============================================================================
# 14. CONDITION ID NORMALIZATION
# =============================================================================

def normalized_condition_series(df):

    if "condition_id" not in df.columns:

        return pd.Series(
            [None] * len(df),
            index=df.index,
            dtype="object"
        )

    values = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )

    invalid = {
        "",
        "nan",
        "none",
        "null",
        "nat"
    }

    values = values.where(
        ~values.str.lower().isin(
            invalid
        ),
        np.nan
    )

    return values


df_a["_normalized_condition_id"] = (
    normalized_condition_series(df_a)
)

df_b["_normalized_condition_id"] = (
    normalized_condition_series(df_b)
)

# =============================================================================
# 15. FROZEN COHORT COVERAGE
# =============================================================================

ids_a = set(
    df_a[
        "_normalized_condition_id"
    ]
    .dropna()
)

ids_b = set(
    df_b[
        "_normalized_condition_id"
    ]
    .dropna()
)

matches_a = (
    frozen_ids
    .intersection(ids_a)
)

matches_b = (
    frozen_ids
    .intersection(ids_b)
)

missing_a = (
    frozen_ids
    - ids_a
)

missing_b = (
    frozen_ids
    - ids_b
)

print("\nFROZEN 1976-CONDITION COVERAGE")
print("-" * 80)

print(
    "Source A matched:",
    len(matches_a),
    "/ 1976"
)

print(
    "Source A missing:",
    len(missing_a)
)

print(
    "Source B matched:",
    len(matches_b),
    "/ 1976"
)

print(
    "Source B missing:",
    len(missing_b)
)

# =============================================================================
# 16. DUPLICATE CONDITION AUDIT
# =============================================================================

print("\nDUPLICATE CONDITION AUDIT")
print("-" * 80)

for source_name, df in {

    "Source A": df_a,
    "Source B": df_b,

}.items():

    subset = df[
        df[
            "_normalized_condition_id"
        ].isin(frozen_ids)
    ]

    counts = (
        subset[
            "_normalized_condition_id"
        ]
        .value_counts()
    )

    duplicated_conditions = int(
        (counts > 1).sum()
    )

    total_rows_in_duplicated_conditions = int(
        counts[
            counts > 1
        ].sum()
    )

    print(
        f"{source_name}:"
    )

    print(
        "  duplicated frozen conditions:",
        duplicated_conditions
    )

    print(
        "  rows belonging to duplicated conditions:",
        total_rows_in_duplicated_conditions
    )

# =============================================================================
# 17. GENOMIC EVIDENCE AUDIT
# =============================================================================

print("\nGENOMIC EVIDENCE AUDIT")
print("-" * 80)

for source_name, df in {

    "Source A": df_a,
    "Source B": df_b,

}.items():

    subset = df[
        df[
            "_normalized_condition_id"
        ].isin(frozen_ids)
    ].copy()

    print(
        f"\n{source_name}"
    )

    if "gene_snp_mutations" in subset.columns:

        mutation_values = (
            subset[
                "gene_snp_mutations"
            ]
            .fillna("")
            .astype(str)
            .str.strip()
        )

        mutation_present = (
            ~mutation_values
            .str.lower()
            .isin(
                [
                    "",
                    "nan",
                    "none",
                    "null"
                ]
            )
        )

        print(
            "  rows with mutation evidence:",
            int(
                mutation_present.sum()
            )
        )

    if (
        "main_lineage" in subset.columns
        and
        "sub_lineage" in subset.columns
    ):

        lineage_present = (
            subset[
                [
                    "main_lineage",
                    "sub_lineage"
                ]
            ]
            .notna()
            .any(axis=1)
        )

        print(
            "  rows with lineage evidence:",
            int(
                lineage_present.sum()
            )
        )

# =============================================================================
# 18. RESISTANCE LABEL AUDIT
# =============================================================================

print("\nRESISTANCE LABEL AUDIT")
print("-" * 80)

for source_name, df in {

    "Source A": df_a,
    "Source B": df_b,

}.items():

    subset = df[
        df[
            "_normalized_condition_id"
        ].isin(frozen_ids)
    ].copy()

    if "drug_resistance_type" not in subset.columns:

        print(
            f"{source_name}: "
            "drug_resistance_type missing"
        )

        continue

    values = (
        subset[
            "drug_resistance_type"
        ]
        .dropna()
        .astype(str)
        .str.strip()
    )

    print(
        f"{source_name} resistance categories:"
    )

    print(
        values.value_counts()
        .to_string()
    )

# =============================================================================
# 19. READ-ONLY SAFETY
# =============================================================================

print("\nREAD-ONLY SAFETY")
print("-" * 80)

print(
    "Source A modified: NO"
)

print(
    "Source B modified: NO"
)

print(
    "Frozen split modified: NO"
)

print(
    "Genomic records modified: NO"
)

print(
    "Vocabulary created: NO"
)

print(
    "Model trained: NO"
)

print(
    "Test predictions: NO"
)

# =============================================================================
# 20. FORENSIC REPORT
# =============================================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as handle:

        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


report = {

    "source_A": str(SOURCE_A),
    "source_B": str(SOURCE_B),

    "source_A_rows":
        int(len(df_a)),

    "source_B_rows":
        int(len(df_b)),

    "frozen_conditions":
        1976,

    "source_A_matches":
        int(len(matches_a)),

    "source_B_matches":
        int(len(matches_b)),

    "source_A_missing":
        int(len(missing_a)),

    "source_B_missing":
        int(len(missing_b)),

    "source_A_sha256":
        sha256_file(SOURCE_A),

    "source_B_sha256":
        sha256_file(SOURCE_B),

    "source_selected":
        False,

    "training_performed":
        False,

    "test_used":
        False,

}

REPORT_FILE = (
    OUTPUT_ROOT /
    "Notebook13_Cell2_Genomic_Source_Forensic_Comparison.json"
)

with open(
    REPORT_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        report,
        handle,
        indent=4
    )

# =============================================================================
# 21. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 2 STATUS: COMPLETE")
print("=" * 80)

print(
    "Source A frozen-condition matches:",
    len(matches_a)
)

print(
    "Source B frozen-condition matches:",
    len(matches_b)
)

print(
    "No genomic source selected automatically."
)

print(
    "No genomic records modified."
)

print(
    "No model training performed."
)

print(
    "\nForensic report:"
)

print(
    REPORT_FILE
)

print(
    "\nSTOP HERE — SOURCE SELECTION MUST BE BASED "
    "ON THIS FORENSIC OUTPUT."
)

NOTEBOOK 13 — CELL 2
GENOMIC SOURCE FORENSIC COMPARISON

PATH GATE
--------------------------------------------------------------------------------
Final split              : PASS
Train split              : PASS
Validation split         : PASS
Test split               : PASS
Source A                 : PASS
Source B                 : PASS

FINAL COHORT GATE
--------------------------------------------------------------------------------
full        : 1976
train       : 1185
validation  : 395
test        : 396
Cohort sizes: PASS

CONDITION UNIQUENESS
--------------------------------------------------------------------------------
full        : 1976
train       : 1185
validation  : 395
test        : 396
Condition uniqueness: PASS

SPLIT OVERLAP
--------------------------------------------------------------------------------
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0
Split integrity: PASS

TARGET DISTRIBUTION
-----------------------------------------------------------------

In [4]:
# =============================================================================
# NOTEBOOK 13 — CELL 3
# FINAL 1976-CONDITION GENOMIC PREPARATION
# RECORD-LEVEL RECONCILIATION AND LEAKAGE SAFETY AUDIT
# =============================================================================

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd

print("=" * 80)
print("NOTEBOOK 13 — CELL 3")
print("FINAL 1976-CONDITION GENOMIC RECORD RECONCILIATION")
print("=" * 80)

# =============================================================================
# 1. AUTHORITATIVE PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT /
    "Step_3B_8_CXR_CoAtNet_Preparation" /
    "QC" /
    "Clean_PSPNet_CXR_Cohort" /
    "FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT /
    "MASTER_CONDITION_LEVEL_SPLIT"
)

GENOMIC_SOURCE = (
    PROJECT_ROOT /
    "TB_Portals_Genomics_March_2025.csv"
)

FULL_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

TRAIN_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_TRAIN.csv"
)

VALIDATION_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT_FILE = (
    SPLIT_ROOT /
    "Notebook08_Cell25_TEST.csv"
)

OUTPUT_ROOT = (
    FINAL_ROOT /
    "Genomic_Preparation_13"
)

CELL3_ROOT = (
    OUTPUT_ROOT /
    "Cell3_Record_Reconciliation"
)

CELL3_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

# =============================================================================
# 2. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Genomic source":
        GENOMIC_SOURCE,

    "Full split":
        FULL_SPLIT_FILE,

    "Train split":
        TRAIN_SPLIT_FILE,

    "Validation split":
        VALIDATION_SPLIT_FILE,

    "Test split":
        TEST_SPLIT_FILE,

}

for name, path in required_paths.items():

    status = path.exists()

    print(
        f"{name:<25}: "
        f"{'PASS' if status else 'FAIL'}"
    )

    if not status:

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

# =============================================================================
# 3. LOAD FROZEN SPLITS
# =============================================================================

full_df = pd.read_csv(
    FULL_SPLIT_FILE
)

train_df = pd.read_csv(
    TRAIN_SPLIT_FILE
)

validation_df = pd.read_csv(
    VALIDATION_SPLIT_FILE
)

test_df = pd.read_csv(
    TEST_SPLIT_FILE
)

for name, df in {

    "full": full_df,
    "train": train_df,
    "validation": validation_df,
    "test": test_df,

}.items():

    required = {
        "condition_id",
        "target_binary"
    }

    missing = (
        required
        - set(df.columns)
    )

    if missing:

        raise RuntimeError(
            f"{name} split missing: "
            f"{sorted(missing)}"
        )

    df["condition_id"] = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )

# =============================================================================
# 4. FROZEN COHORT GATE
# =============================================================================

frozen_ids = set(
    full_df["condition_id"]
)

train_ids = set(
    train_df["condition_id"]
)

validation_ids = set(
    validation_df["condition_id"]
)

test_ids = set(
    test_df["condition_id"]
)

print("\nFROZEN COHORT GATE")
print("-" * 80)

print(
    "Full:",
    len(frozen_ids)
)

print(
    "Train:",
    len(train_ids)
)

print(
    "Validation:",
    len(validation_ids)
)

print(
    "Test:",
    len(test_ids)
)

if len(frozen_ids) != 1976:
    raise RuntimeError(
        "Frozen cohort is not 1976."
    )

if len(train_ids) != 1185:
    raise RuntimeError(
        "Train split is not 1185."
    )

if len(validation_ids) != 395:
    raise RuntimeError(
        "Validation split is not 395."
    )

if len(test_ids) != 396:
    raise RuntimeError(
        "Test split is not 396."
    )

if (
    train_ids & validation_ids
    or
    train_ids & test_ids
    or
    validation_ids & test_ids
):

    raise RuntimeError(
        "Condition overlap detected."
    )

print(
    "Frozen cohort and split integrity: PASS"
)

# =============================================================================
# 5. LOAD AUTHORITATIVE GENOMIC SOURCE
# =============================================================================

genomic_df = pd.read_csv(
    GENOMIC_SOURCE,
    low_memory=False
)

print("\nGENOMIC SOURCE")
print("-" * 80)

print(
    "Total genomic records:",
    len(genomic_df)
)

if len(genomic_df) != 4467:

    print(
        "WARNING: Source row count differs "
        "from previously audited 4467."
    )

# =============================================================================
# 6. REQUIRED GENOMIC SCHEMA
# =============================================================================

required_genomic_columns = [

    "condition_id",
    "main_lineage",
    "sub_lineage",
    "drug_resistance_type",
    "gene_snp_mutations",

]

missing = [
    c
    for c in required_genomic_columns
    if c not in genomic_df.columns
]

if missing:

    raise RuntimeError(
        "Required genomic columns missing: "
        f"{missing}"
    )

print(
    "Required genomic schema: PASS"
)

# =============================================================================
# 7. NORMALIZE CONDITION IDS
# =============================================================================

genomic_df["_condition_id"] = (
    genomic_df["condition_id"]
    .astype(str)
    .str.strip()
)

genomic_df = genomic_df[
    genomic_df["_condition_id"].notna()
].copy()

# =============================================================================
# 8. EXTRACT ONLY FINAL 1976 CONDITIONS
# =============================================================================

final_genomic = genomic_df[
    genomic_df["_condition_id"].isin(
        frozen_ids
    )
].copy()

print("\nFINAL COHORT GENOMIC EXTRACTION")
print("-" * 80)

print(
    "Final genomic rows:",
    len(final_genomic)
)

matched_conditions = (
    final_genomic[
        "_condition_id"
    ]
    .nunique()
)

print(
    "Matched conditions:",
    matched_conditions
)

if matched_conditions != 1976:

    missing_conditions = (
        frozen_ids
        -
        set(
            final_genomic[
                "_condition_id"
            ]
        )
    )

    raise RuntimeError(
        f"Not all 1976 conditions have genomic "
        f"records. Missing: "
        f"{len(missing_conditions)}"
    )

print(
    "All 1976 conditions represented: PASS"
)

# =============================================================================
# 9. RECORD COUNTS PER CONDITION
# =============================================================================

record_counts = (
    final_genomic[
        "_condition_id"
    ]
    .value_counts()
    .rename(
        "genomic_record_count"
    )
)

condition_audit = (
    full_df[
        [
            "condition_id",
            "target_binary"
        ]
    ]
    .copy()
)

condition_audit = condition_audit.merge(
    record_counts,
    left_on="condition_id",
    right_index=True,
    how="left"
)

condition_audit[
    "genomic_record_count"
] = (
    condition_audit[
        "genomic_record_count"
    ]
    .fillna(0)
    .astype(int)
)

print("\nRECORD COUNT AUDIT")
print("-" * 80)

print(
    "Conditions with exactly 1 genomic record:",
    int(
        (
            condition_audit[
                "genomic_record_count"
            ] == 1
        ).sum()
    )
)

print(
    "Conditions with >1 genomic record:",
    int(
        (
            condition_audit[
                "genomic_record_count"
            ] > 1
        ).sum()
    )
)

print(
    "Maximum records for one condition:",
    int(
        condition_audit[
            "genomic_record_count"
        ].max()
    )
)

# =============================================================================
# 10. SPECIMEN-LEVEL ID DISCOVERY
# =============================================================================

print("\nSPECIMEN IDENTIFIER AUDIT")
print("-" * 80)

possible_specimen_columns = [

    "specimen_id",
    "specimen_identifier",
    "specimen",
    "sample_id",
    "sample_identifier",

]

available_specimen_columns = [
    c
    for c in possible_specimen_columns
    if c in final_genomic.columns
]

print(
    "Available specimen columns:",
    available_specimen_columns
)

if not available_specimen_columns:

    print(
        "No standard specimen identifier column "
        "was found."
    )

# =============================================================================
# 11. CREATE INTERNAL RECORD ID
# =============================================================================

if "specimen_id" in final_genomic.columns:

    specimen_series = (
        final_genomic[
            "specimen_id"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

elif "specimen_identifier" in final_genomic.columns:

    specimen_series = (
        final_genomic[
            "specimen_identifier"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
    )

else:

    specimen_series = pd.Series(
        "",
        index=final_genomic.index
    )

final_genomic[
    "_specimen_key"
] = specimen_series

# =============================================================================
# 12. CONDITION + SPECIMEN DUPLICATION
# =============================================================================

record_key = (
    final_genomic[
        "_condition_id"
    ]
    .astype(str)
    +
    "||"
    +
    final_genomic[
        "_specimen_key"
    ]
    .astype(str)
)

final_genomic[
    "_condition_specimen_key"
] = record_key

duplicate_condition_specimen = (
    final_genomic[
        "_condition_specimen_key"
    ]
    .duplicated(
        keep=False
    )
)

duplicate_rows = int(
    duplicate_condition_specimen.sum()
)

duplicate_groups = int(
    final_genomic.loc[
        duplicate_condition_specimen,
        "_condition_specimen_key"
    ]
    .nunique()
)

print("\nCONDITION + SPECIMEN DUPLICATION")
print("-" * 80)

print(
    "Duplicate condition/specimen rows:",
    duplicate_rows
)

print(
    "Duplicate condition/specimen groups:",
    duplicate_groups
)

# =============================================================================
# 13. RESISTANCE LABEL CONSISTENCY
# =============================================================================

print("\nRESISTANCE LABEL CONSISTENCY")
print("-" * 80)

if "drug_resistance_type" not in final_genomic.columns:

    raise RuntimeError(
        "drug_resistance_type missing."
    )

label_check = (
    final_genomic[
        [
            "_condition_id",
            "drug_resistance_type"
        ]
    ]
    .copy()
)

label_check[
    "drug_resistance_type"
] = (
    label_check[
        "drug_resistance_type"
    ]
    .fillna(
        ""
    )
    .astype(str)
    .str.strip()
)

label_groups = (
    label_check
    .groupby(
        "_condition_id"
    )[
        "drug_resistance_type"
    ]
    .apply(
        lambda x: sorted(
            set(
                v
                for v in x
                if v != ""
            )
        )
    )
)

label_conflicts = (
    label_groups[
        label_groups.apply(
            len
        ) > 1
    ]
)

print(
    "Conditions with >1 "
    "drug_resistance_type:",
    len(label_conflicts)
)

if len(label_conflicts) > 0:

    print(
        "\nConflicting examples:"
    )

    print(
        label_conflicts
        .head(20)
        .to_string()
    )

# =============================================================================
# 14. FROZEN TARGET VS GENOMIC RESISTANCE CATEGORY
# =============================================================================

print("\nFROZEN TARGET VS GENOMIC CATEGORY")
print("-" * 80)

target_map = (
    full_df[
        [
            "condition_id",
            "target_binary"
        ]
    ]
    .set_index(
        "condition_id"
    )[
        "target_binary"
    ]
    .to_dict()
)

final_genomic[
    "_frozen_target"
] = (
    final_genomic[
        "_condition_id"
    ]
    .map(
        target_map
    )
)

# Known source categories from the audited March 2025 dataset.
# These are descriptive categories only.
sensitive_categories = {
    "Sensitive",
    "Sensitive-TB",
    "Drug Sensitive",
    "Drug-Sensitive",
}

resistant_categories = {
    "MDR-TB",
    "Pre-XDR-TB",
    "XDR-TB",
    "RR-TB",
    "HR-TB",
    "Mono-DR",
    "Poly-DR",
    "MDR non-XDR",
    "Pre-XDR",
}

def resistance_class(value):

    if pd.isna(value):

        return "MISSING"

    value = str(
        value
    ).strip()

    if value in sensitive_categories:

        return 0

    if value in resistant_categories:

        return 1

    return -1


final_genomic[
    "_derived_resistance_class"
] = (
    final_genomic[
        "drug_resistance_type"
    ]
    .apply(
        resistance_class
    )
)

# =============================================================================
# 15. TARGET CONFLICT AUDIT
# =============================================================================

known_category_rows = (
    final_genomic[
        "_derived_resistance_class"
    ].isin(
        [0, 1]
    )
)

target_conflicts = (
    known_category_rows
    &
    (
        final_genomic[
            "_derived_resistance_class"
        ]
        !=
        final_genomic[
            "_frozen_target"
        ]
    )
)

print(
    "Rows with known resistance category:",
    int(
        known_category_rows.sum()
    )
)

print(
    "Rows conflicting with frozen target:",
    int(
        target_conflicts.sum()
    )
)

conflicting_conditions = (
    final_genomic.loc[
        target_conflicts,
        "_condition_id"
    ]
    .nunique()
)

print(
    "Conditions with target/category conflict:",
    conflicting_conditions
)

# =============================================================================
# 16. GENOMIC EVIDENCE COVERAGE
# =============================================================================

print("\nGENOMIC EVIDENCE COVERAGE")
print("-" * 80)

mutation_text = (
    final_genomic[
        "gene_snp_mutations"
    ]
    .fillna("")
    .astype(str)
    .str.strip()
)

mutation_present = (
    ~mutation_text
    .str.lower()
    .isin(
        [
            "",
            "nan",
            "none",
            "null",
        ]
    )
)

mutation_conditions = (
    final_genomic.loc[
        mutation_present,
        "_condition_id"
    ]
    .nunique()
)

print(
    "Conditions with mutation evidence:",
    mutation_conditions,
    "/ 1976"
)

if (
    "main_lineage" in final_genomic.columns
    and
    "sub_lineage" in final_genomic.columns
):

    lineage_present = (
        final_genomic[
            [
                "main_lineage",
                "sub_lineage"
            ]
        ]
        .notna()
        .any(axis=1)
    )

    lineage_conditions = (
        final_genomic.loc[
            lineage_present,
            "_condition_id"
        ]
        .nunique()
    )

else:

    lineage_conditions = 0

print(
    "Conditions with lineage evidence:",
    lineage_conditions,
    "/ 1976"
)

# =============================================================================
# 17. TARGET-SPECIFIC MISSINGNESS AUDIT
# =============================================================================

print("\nTARGET-SPECIFIC GENOMIC EVIDENCE")
print("-" * 80)

condition_evidence = (
    final_genomic
    .groupby(
        "_condition_id"
    )
    .agg(
        mutation_evidence=(
            "_condition_id",
            lambda x: False
        )
    )
)

# Build condition-level mutation evidence correctly.
mutation_condition_set = set(
    final_genomic.loc[
        mutation_present,
        "_condition_id"
    ]
)

condition_evidence = pd.DataFrame(
    {
        "condition_id":
            list(frozen_ids)
    }
)

condition_evidence[
    "target_binary"
] = (
    condition_evidence[
        "condition_id"
    ]
    .map(
        target_map
    )
)

condition_evidence[
    "mutation_evidence"
] = (
    condition_evidence[
        "condition_id"
    ]
    .isin(
        mutation_condition_set
    )
)

print(
    pd.crosstab(
        condition_evidence[
            "target_binary"
        ],
        condition_evidence[
            "mutation_evidence"
        ]
    )
    .rename(
        columns={
            False: "No mutation evidence",
            True: "Mutation evidence",
        },
        index={
            0: "DS-TB",
            1: "DR-TB",
        }
    )
    .to_string()
)

# =============================================================================
# 18. SAVE CONDITION AUDIT
# =============================================================================

condition_audit_file = (
    CELL3_ROOT /
    "Notebook13_Cell3_Condition_Level_Genomic_Record_Audit.csv"
)

condition_audit.to_csv(
    condition_audit_file,
    index=False
)

# =============================================================================
# 19. SAVE MULTI-RECORD CONDITIONS
# =============================================================================

multi_record_conditions = (
    condition_audit[
        condition_audit[
            "genomic_record_count"
        ] > 1
    ]
    .copy()
)

multi_record_file = (
    CELL3_ROOT /
    "Notebook13_Cell3_MultiRecord_Conditions.csv"
)

multi_record_conditions.to_csv(
    multi_record_file,
    index=False
)

# =============================================================================
# 20. SAVE CONFLICTING CONDITIONS
# =============================================================================

conflict_file = (
    CELL3_ROOT /
    "Notebook13_Cell3_Resistance_Label_Conflicts.csv"
)

if len(label_conflicts) > 0:

    conflict_rows = (
        final_genomic[
            final_genomic[
                "_condition_id"
            ].isin(
                set(
                    label_conflicts.index
                )
            )
        ]
        .copy()
    )

else:

    conflict_rows = pd.DataFrame()

conflict_rows.to_csv(
    conflict_file,
    index=False
)

# =============================================================================
# 21. SAVE TARGET-CONFLICT RECORDS
# =============================================================================

target_conflict_file = (
    CELL3_ROOT /
    "Notebook13_Cell3_Target_Category_Conflicts.csv"
)

final_genomic.loc[
    target_conflicts
].to_csv(
    target_conflict_file,
    index=False
)

# =============================================================================
# 22. SAVE FULL EXTRACTED GENOMIC RECORDS
# =============================================================================

# This is a COPY only.
# The original TB Portals source is never modified.

extracted_file = (
    CELL3_ROOT /
    "Notebook13_Cell3_FINAL_1976_Genomic_Records_Unresolved.csv"
)

final_genomic.to_csv(
    extracted_file,
    index=False
)

# =============================================================================
# 23. SUMMARY
# =============================================================================

summary = {

    "notebook":
        "13_Final_1976_Genomic_Preparation",

    "cell":
        "Cell3_Record_Reconciliation",

    "authoritative_source":
        str(GENOMIC_SOURCE),

    "source_rows":
        int(len(genomic_df)),

    "final_conditions":
        1976,

    "final_genomic_rows":
        int(len(final_genomic)),

    "conditions_with_genomic_records":
        int(matched_conditions),

    "conditions_with_multiple_records":
        int(
            (
                condition_audit[
                    "genomic_record_count"
                ] > 1
            ).sum()
        ),

    "duplicate_condition_specimen_rows":
        duplicate_rows,

    "duplicate_condition_specimen_groups":
        duplicate_groups,

    "conditions_with_multiple_resistance_labels":
        int(len(label_conflicts)),

    "rows_conflicting_with_frozen_target":
        int(target_conflicts.sum()),

    "conditions_conflicting_with_frozen_target":
        int(conflicting_conditions),

    "conditions_with_mutation_evidence":
        int(mutation_conditions),

    "conditions_with_lineage_evidence":
        int(lineage_conditions),

    "training_performed":
        False,

    "test_predictions":
        False,

    "source_modified":
        False,

    "status":
        "AUDIT_COMPLETE_REQUIRES_REVIEW_BEFORE_REDUCTION"

}

SUMMARY_FILE = (
    CELL3_ROOT /
    "Notebook13_Cell3_Genomic_Reconciliation_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        summary,
        handle,
        indent=4
    )

# =============================================================================
# 24. FINAL OUTPUT
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 3 STATUS: AUDIT COMPLETE")
print("=" * 80)

print(
    "Final conditions:",
    1976
)

print(
    "Genomic records:",
    len(final_genomic)
)

print(
    "Multi-record conditions:",
    int(
        (
            condition_audit[
                "genomic_record_count"
            ] > 1
        ).sum()
    )
)

print(
    "Resistance-label conflicts:",
    len(label_conflicts)
)

print(
    "Target/category conflicts:",
    conflicting_conditions
)

print(
    "Mutation-evidence conditions:",
    mutation_conditions
)

print(
    "Lineage-evidence conditions:",
    lineage_conditions
)

print(
    "\nNo training performed."
)

print(
    "No test predictions performed."
)

print(
    "No source data modified."
)

print(
    "\nAudit summary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nIMPORTANT:"
)

print(
    "DO NOT proceed to tokenization/training until "
    "the multi-record and conflict results are reviewed."
)

NOTEBOOK 13 — CELL 3
FINAL 1976-CONDITION GENOMIC RECORD RECONCILIATION

PATH GATE
--------------------------------------------------------------------------------
Genomic source           : PASS
Full split               : PASS
Train split              : PASS
Validation split         : PASS
Test split               : PASS

FROZEN COHORT GATE
--------------------------------------------------------------------------------
Full: 1976
Train: 1185
Validation: 395
Test: 396
Frozen cohort and split integrity: PASS

GENOMIC SOURCE
--------------------------------------------------------------------------------
Total genomic records: 4467
Required genomic schema: PASS

FINAL COHORT GENOMIC EXTRACTION
--------------------------------------------------------------------------------
Final genomic rows: 2128
Matched conditions: 1976
All 1976 conditions represented: PASS

RECORD COUNT AUDIT
--------------------------------------------------------------------------------
Conditions with exactly 1 ge

In [5]:
# =============================================================================
# NOTEBOOK 13 — CELL 4
# FINAL 1976-CONDITION GENOMIC PREPARATION
# CONDITION-LEVEL GENOMIC RECONCILIATION
# =============================================================================

from pathlib import Path
import json
import re
import hashlib

import numpy as np
import pandas as pd


print("=" * 80)
print("NOTEBOOK 13 — CELL 4")
print("FINAL CONDITION-LEVEL GENOMIC RECONCILIATION")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

GENOMIC_SOURCE = (
    PROJECT_ROOT
    / "TB_Portals_Genomics_March_2025.csv"
)

FULL_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

TRAIN_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TRAIN.csv"
)

VALIDATION_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TEST.csv"
)

CELL4_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell4_Condition_Level_Reconciliation"
)

CELL4_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# 2. LOAD FROZEN SPLITS
# =============================================================================

full_df = pd.read_csv(
    FULL_SPLIT_FILE
)

train_df = pd.read_csv(
    TRAIN_SPLIT_FILE
)

validation_df = pd.read_csv(
    VALIDATION_SPLIT_FILE
)

test_df = pd.read_csv(
    TEST_SPLIT_FILE
)


for df in [
    full_df,
    train_df,
    validation_df,
    test_df
]:

    df["condition_id"] = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )


frozen_ids = set(
    full_df["condition_id"]
)

train_ids = set(
    train_df["condition_id"]
)

validation_ids = set(
    validation_df["condition_id"]
)

test_ids = set(
    test_df["condition_id"]
)


# =============================================================================
# 3. FINAL SPLIT SAFETY
# =============================================================================

print("\nFROZEN SPLIT SAFETY")
print("-" * 80)

print(
    "Full:",
    len(frozen_ids)
)

print(
    "Train:",
    len(train_ids)
)

print(
    "Validation:",
    len(validation_ids)
)

print(
    "Test:",
    len(test_ids)
)

if len(frozen_ids) != 1976:
    raise RuntimeError(
        "Frozen cohort must contain exactly 1976 conditions."
    )

if len(train_ids) != 1185:
    raise RuntimeError(
        "Train split must contain exactly 1185 conditions."
    )

if len(validation_ids) != 395:
    raise RuntimeError(
        "Validation split must contain exactly 395 conditions."
    )

if len(test_ids) != 396:
    raise RuntimeError(
        "Test split must contain exactly 396 conditions."
    )

if train_ids & validation_ids:
    raise RuntimeError(
        "Train/validation overlap detected."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test overlap detected."
    )

if validation_ids & test_ids:
    raise RuntimeError(
        "Validation/test overlap detected."
    )

print(
    "Split integrity: PASS"
)


# =============================================================================
# 4. LOAD AUTHORITATIVE GENOMIC SOURCE
# =============================================================================

genomic_df = pd.read_csv(
    GENOMIC_SOURCE,
    low_memory=False
)

print("\nGENOMIC SOURCE")
print("-" * 80)

print(
    "Source rows:",
    len(genomic_df)
)

if len(genomic_df) != 4467:
    raise RuntimeError(
        f"Expected 4467 source rows, "
        f"found {len(genomic_df)}."
    )


# =============================================================================
# 5. REQUIRED INPUT COLUMNS
# =============================================================================

required_columns = [
    "condition_id",
    "main_lineage",
    "sub_lineage",
    "gene_snp_mutations",
]

missing = [
    c
    for c in required_columns
    if c not in genomic_df.columns
]

if missing:

    raise RuntimeError(
        "Required genomic input columns missing: "
        f"{missing}"
    )

print(
    "Genomic input schema: PASS"
)


# =============================================================================
# 6. TARGET/RESISTANCE FIELDS — EXPLICITLY EXCLUDED
# =============================================================================

excluded_target_fields = [

    "type_of_resistance",
    "drug_resistance_type",
    "num_drug_resistant_variants",

]

print("\nTARGET-DERIVED FIELD EXCLUSION")
print("-" * 80)

for column in excluded_target_fields:

    if column in genomic_df.columns:

        print(
            f"{column:<35}: EXCLUDED"
        )

    else:

        print(
            f"{column:<35}: NOT PRESENT"
        )

print(
    "Resistance-label fields will NOT become "
    "Transformer inputs."
)


# =============================================================================
# 7. EXTRACT FINAL 1976 CONDITIONS
# =============================================================================

genomic_df["condition_id"] = (
    genomic_df["condition_id"]
    .astype(str)
    .str.strip()
)

final_records = genomic_df[
    genomic_df["condition_id"].isin(
        frozen_ids
    )
].copy()

print("\nFINAL GENOMIC EXTRACTION")
print("-" * 80)

print(
    "Rows:",
    len(final_records)
)

print(
    "Conditions:",
    final_records[
        "condition_id"
    ].nunique()
)

if (
    len(
        final_records[
            "condition_id"
        ].unique()
    )
    != 1976
):

    raise RuntimeError(
        "Final genomic records do not cover "
        "all 1976 conditions."
    )

print(
    "1976-condition coverage: PASS"
)


# =============================================================================
# 8. STANDARDIZE MISSING VALUES
# =============================================================================

NULL_STRINGS = {
    "",
    "nan",
    "none",
    "null",
    "nat",
    "na",
    "n/a"
}


def clean_text(value):

    if pd.isna(value):
        return ""

    value = str(
        value
    ).strip()

    if value.lower() in NULL_STRINGS:
        return ""

    return value


for column in [
    "main_lineage",
    "sub_lineage",
    "gene_snp_mutations",
]:

    final_records[column] = (
        final_records[column]
        .apply(clean_text)
    )


# =============================================================================
# 9. IDENTIFY SPECIMEN FIELDS
# =============================================================================

specimen_columns = [

    "specimen_id",
    "specimen_identifier",

]

available_specimen_columns = [
    c
    for c in specimen_columns
    if c in final_records.columns
]

print("\nSPECIMEN FIELDS")
print("-" * 80)

print(
    "Available:",
    available_specimen_columns
)


# =============================================================================
# 10. EXACT DUPLICATE AUDIT
# =============================================================================

# Only fields that are legitimate genomic representation fields
# are used for exact duplicate detection.

dedup_columns = [
    "condition_id",
    "main_lineage",
    "sub_lineage",
    "gene_snp_mutations",
]

dedup_columns = [
    c
    for c in dedup_columns
    if c in final_records.columns
]

before_dedup = len(
    final_records
)

exact_duplicate_mask = (
    final_records[
        dedup_columns
    ]
    .duplicated(
        keep="first"
    )
)

exact_duplicate_count = int(
    exact_duplicate_mask.sum()
)

final_records_dedup = (
    final_records[
        ~exact_duplicate_mask
    ]
    .copy()
)

after_dedup = len(
    final_records_dedup
)

print("\nEXACT DUPLICATE RECONCILIATION")
print("-" * 80)

print(
    "Rows before:",
    before_dedup
)

print(
    "Exact duplicate rows removed:",
    exact_duplicate_count
)

print(
    "Rows after:",
    after_dedup
)

print(
    "Remaining unique genomic records:",
    after_dedup
)


# =============================================================================
# 11. CONDITION COVERAGE AFTER DEDUPLICATION
# =============================================================================

remaining_conditions = set(
    final_records_dedup[
        "condition_id"
    ]
)

if remaining_conditions != frozen_ids:

    missing_after_dedup = (
        frozen_ids
        - remaining_conditions
    )

    unexpected_after_dedup = (
        remaining_conditions
        - frozen_ids
    )

    raise RuntimeError(
        "Condition identity changed after "
        "deduplication.\n"
        f"Missing: {len(missing_after_dedup)}\n"
        f"Unexpected: {len(unexpected_after_dedup)}"
    )

print(
    "1976-condition identity after deduplication: PASS"
)


# =============================================================================
# 12. BUILD CONDITION-LEVEL GENOMIC REPRESENTATION
# =============================================================================

def split_mutation_tokens(text):

    if not text:
        return []

    text = str(
        text
    ).strip()

    if not text:
        return []

    # The source may contain multiple mutation
    # representations separated by common delimiters.
    pieces = re.split(
        r"[;,|]+",
        text
    )

    tokens = []

    for piece in pieces:

        piece = piece.strip()

        if not piece:
            continue

        tokens.append(
            piece
        )

    return tokens


def unique_preserve_order(values):

    seen = set()
    output = []

    for value in values:

        normalized = (
            str(value)
            .strip()
        )

        if not normalized:
            continue

        key = normalized.lower()

        if key not in seen:

            seen.add(key)
            output.append(
                normalized
            )

    return output


# -------------------------------------------------------------------------
# Aggregate every genomic record belonging to a condition.
#
# We preserve:
#   - all unique mutation evidence
#   - all unique lineage evidence
#
# We do NOT aggregate:
#   - drug_resistance_type
#   - type_of_resistance
#   - num_drug_resistant_variants
#
# Those fields are target/proxy information and are excluded.
# -------------------------------------------------------------------------

condition_rows = []

for condition_id, group in final_records_dedup.groupby(
    "condition_id",
    sort=False
):

    main_lineages = unique_preserve_order(
        group[
            "main_lineage"
        ]
        .tolist()
    )

    sub_lineages = unique_preserve_order(
        group[
            "sub_lineage"
        ]
        .tolist()
    )

    mutation_tokens = []

    for value in group[
        "gene_snp_mutations"
    ]:

        mutation_tokens.extend(
            split_mutation_tokens(
                value
            )
        )

    mutation_tokens = (
        unique_preserve_order(
            mutation_tokens
        )
    )

    condition_rows.append(
        {
            "condition_id":
                condition_id,

            "genomic_record_count":
                int(len(group)),

            "main_lineage":
                " | ".join(
                    main_lineages
                ),

            "sub_lineage":
                " | ".join(
                    sub_lineages
                ),

            "mutation_token_count":
                int(
                    len(
                        mutation_tokens
                    )
                ),

            "gene_snp_mutations_aggregated":
                " | ".join(
                    mutation_tokens
                ),

        }
    )


condition_genomic_df = pd.DataFrame(
    condition_rows
)


# =============================================================================
# 13. CONDITION-LEVEL IDENTITY GATE
# =============================================================================

print("\nCONDITION-LEVEL AGGREGATION")
print("-" * 80)

print(
    "Aggregated conditions:",
    len(condition_genomic_df)
)

print(
    "Unique condition IDs:",
    condition_genomic_df[
        "condition_id"
    ].nunique()
)

if len(condition_genomic_df) != 1976:

    raise RuntimeError(
        "Condition-level aggregation did not "
        "produce exactly 1976 conditions."
    )

if (
    condition_genomic_df[
        "condition_id"
    ].nunique()
    != 1976
):

    raise RuntimeError(
        "Duplicate condition IDs after aggregation."
    )

print(
    "Condition-level identity: PASS"
)


# =============================================================================
# 14. MERGE FROZEN TARGET
# =============================================================================

condition_genomic_df = condition_genomic_df.merge(
    full_df[
        [
            "condition_id",
            "target_binary"
        ]
    ],
    on="condition_id",
    how="left",
    validate="one_to_one"
)


if (
    condition_genomic_df[
        "target_binary"
    ].isna().any()
):

    raise RuntimeError(
        "Some genomic conditions do not have "
        "a frozen target."
    )


# =============================================================================
# 15. TARGET PRESERVATION
# =============================================================================

target_counts = (
    condition_genomic_df[
        "target_binary"
    ]
    .value_counts()
    .to_dict()
)

print("\nTARGET PRESERVATION")
print("-" * 80)

print(
    "DS:",
    int(
        target_counts.get(
            0,
            0
        )
    )
)

print(
    "DR:",
    int(
        target_counts.get(
            1,
            0
        )
    )
)

if (
    int(target_counts.get(0, 0)) != 704
    or
    int(target_counts.get(1, 0)) != 1272
):

    raise RuntimeError(
        "Frozen target distribution changed."
    )

print(
    "Frozen target preservation: PASS"
)


# =============================================================================
# 16. GENOMIC EVIDENCE COVERAGE
# =============================================================================

condition_genomic_df[
    "has_mutation_evidence"
] = (
    condition_genomic_df[
        "mutation_token_count"
    ]
    > 0
)

condition_genomic_df[
    "has_lineage_evidence"
] = (
    (
        condition_genomic_df[
            "main_lineage"
        ].str.strip()
        != ""
    )
    |
    (
        condition_genomic_df[
            "sub_lineage"
        ].str.strip()
        != ""
    )
)

print("\nGENOMIC EVIDENCE AFTER AGGREGATION")
print("-" * 80)

print(
    "Mutation evidence:",
    int(
        condition_genomic_df[
            "has_mutation_evidence"
        ].sum()
    ),
    "/ 1976"
)

print(
    "Lineage evidence:",
    int(
        condition_genomic_df[
            "has_lineage_evidence"
        ].sum()
    ),
    "/ 1976"
)


# =============================================================================
# 17. TARGET-SPECIFIC MISSINGNESS
# =============================================================================

print("\nTARGET-SPECIFIC MUTATION EVIDENCE")
print("-" * 80)

missingness_table = pd.crosstab(
    condition_genomic_df[
        "target_binary"
    ],
    condition_genomic_df[
        "has_mutation_evidence"
    ]
)

missingness_table = (
    missingness_table
    .rename(
        index={
            0: "DS-TB",
            1: "DR-TB"
        },
        columns={
            False:
                "No mutation evidence",
            True:
                "Mutation evidence"
        }
    )
)

print(
    missingness_table
)


# =============================================================================
# 18. MULTI-RECORD CONDITION DISTRIBUTION
# =============================================================================

print("\nMULTI-RECORD CONDITION DISTRIBUTION")
print("-" * 80)

multi_record_distribution = (
    condition_genomic_df[
        "genomic_record_count"
    ]
    .value_counts()
    .sort_index()
)

print(
    multi_record_distribution.to_string()
)


# =============================================================================
# 19. INPUT SAFETY AUDIT
# =============================================================================

print("\nINPUT SAFETY AUDIT")
print("-" * 80)

for forbidden in [
    "drug_resistance_type",
    "type_of_resistance",
    "num_drug_resistant_variants",
]:

    if forbidden in condition_genomic_df.columns:

        raise RuntimeError(
            f"Forbidden target-derived field "
            f"present in final representation: "
            f"{forbidden}"
        )

print(
    "Target-derived resistance fields: EXCLUDED"
)

print(
    "Frozen target used only as label: PASS"
)


# =============================================================================
# 20. SAVE FINAL CONDITION-LEVEL GENOMIC DATASET
# =============================================================================

FINAL_GENOMIC_FILE = (
    CELL4_ROOT /
    "Notebook13_Cell4_FINAL_1976_Condition_Level_Genomic_Dataset.csv"
)

condition_genomic_df.to_csv(
    FINAL_GENOMIC_FILE,
    index=False
)


# =============================================================================
# 21. SAVE DEDUPLICATED RECORD-LEVEL DATA
# =============================================================================

DEDUP_RECORD_FILE = (
    CELL4_ROOT /
    "Notebook13_Cell4_FINAL_Deduplicated_Genomic_Records.csv"
)

final_records_dedup.to_csv(
    DEDUP_RECORD_FILE,
    index=False
)


# =============================================================================
# 22. SAVE RECONCILIATION SUMMARY
# =============================================================================

summary = {

    "source":
        str(GENOMIC_SOURCE),

    "source_rows":
        int(len(genomic_df)),

    "final_raw_genomic_rows":
        int(len(final_records)),

    "exact_duplicate_rows_removed":
        exact_duplicate_count,

    "final_deduplicated_genomic_rows":
        int(len(final_records_dedup)),

    "final_condition_count":
        int(len(condition_genomic_df)),

    "single_record_conditions":
        int(
            (
                condition_genomic_df[
                    "genomic_record_count"
                ] == 1
            ).sum()
        ),

    "multi_record_conditions":
        int(
            (
                condition_genomic_df[
                    "genomic_record_count"
                ] > 1
            ).sum()
        ),

    "max_records_per_condition":
        int(
            condition_genomic_df[
                "genomic_record_count"
            ].max()
        ),

    "mutation_evidence_conditions":
        int(
            condition_genomic_df[
                "has_mutation_evidence"
            ].sum()
        ),

    "lineage_evidence_conditions":
        int(
            condition_genomic_df[
                "has_lineage_evidence"
            ].sum()
        ),

    "ds_conditions":
        704,

    "dr_conditions":
        1272,

    "target_derived_fields_excluded":
        excluded_target_fields,

    "training_performed":
        False,

    "validation_used_for_training":
        False,

    "test_used_for_training":
        False,

    "test_predictions":
        False,

    "source_modified":
        False,

    "status":
        "PASS_CONDITION_LEVEL_DATASET_BUILT"

}

SUMMARY_FILE = (
    CELL4_ROOT /
    "Notebook13_Cell4_Reconciliation_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as handle:

    json.dump(
        summary,
        handle,
        indent=4
    )


# =============================================================================
# 23. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 4 STATUS: PASS")
print("=" * 80)

print(
    "Final condition-level genomic records:",
    len(condition_genomic_df)
)

print(
    "Expected:",
    1976
)

print(
    "Exact duplicate rows removed:",
    exact_duplicate_count
)

print(
    "Conditions with genomic records:",
    len(condition_genomic_df)
)

print(
    "Mutation evidence conditions:",
    int(
        condition_genomic_df[
            "has_mutation_evidence"
        ].sum()
    )
)

print(
    "Lineage evidence conditions:",
    int(
        condition_genomic_df[
            "has_lineage_evidence"
        ].sum()
    )
)

print(
    "Target-derived resistance fields:",
    "EXCLUDED"
)

print(
    "Training:",
    "NOT PERFORMED"
)

print(
    "Test evaluation:",
    "NOT PERFORMED"
)

print(
    "\nFinal genomic dataset:"
)

print(
    FINAL_GENOMIC_FILE
)

print(
    "\nReconciliation summary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nProceed to Cell 5 only if STATUS = PASS."
)

NOTEBOOK 13 — CELL 4
FINAL CONDITION-LEVEL GENOMIC RECONCILIATION

FROZEN SPLIT SAFETY
--------------------------------------------------------------------------------
Full: 1976
Train: 1185
Validation: 395
Test: 396
Split integrity: PASS

GENOMIC SOURCE
--------------------------------------------------------------------------------
Source rows: 4467
Genomic input schema: PASS

TARGET-DERIVED FIELD EXCLUSION
--------------------------------------------------------------------------------
type_of_resistance                 : EXCLUDED
drug_resistance_type               : EXCLUDED
num_drug_resistant_variants        : EXCLUDED
Resistance-label fields will NOT become Transformer inputs.

FINAL GENOMIC EXTRACTION
--------------------------------------------------------------------------------
Rows: 2128
Conditions: 1976
1976-condition coverage: PASS

SPECIMEN FIELDS
--------------------------------------------------------------------------------
Available: ['specimen_id', 'specimen_identifi

In [7]:
# =============================================================================
# NOTEBOOK 13 — CELL 5
# TRAINING-ONLY GENOMIC VOCABULARY
# AND FINAL 1976 CONDITION SEQUENCE PREPARATION
# =============================================================================

from pathlib import Path
import json
import re
import hashlib
from collections import Counter

import numpy as np
import pandas as pd


print("=" * 80)
print("NOTEBOOK 13 — CELL 5")
print("TRAINING-ONLY GENOMIC VOCABULARY AND SEQUENCE PREPARATION")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

SPLIT_ROOT = (
    FINAL_ROOT
    / "MASTER_CONDITION_LEVEL_SPLIT"
)

CELL4_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell4_Condition_Level_Reconciliation"
)

CELL5_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell5_Training_Only_Vocabulary"
)

CELL5_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_GENOMIC_FILE = (
    CELL4_ROOT
    / "Notebook13_Cell4_FINAL_1976_Condition_Level_Genomic_Dataset.csv"
)

FULL_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_FINAL_1976_Condition_Level_Split.csv"
)

TRAIN_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TRAIN.csv"
)

VALIDATION_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_VALIDATION.csv"
)

TEST_SPLIT_FILE = (
    SPLIT_ROOT
    / "Notebook08_Cell25_TEST.csv"
)


# =============================================================================
# 2. LOAD DATA
# =============================================================================

print("\nDATA LOADING")
print("-" * 80)

genomic_df = pd.read_csv(
    FINAL_GENOMIC_FILE,
    low_memory=False
)

full_df = pd.read_csv(
    FULL_SPLIT_FILE
)

train_df = pd.read_csv(
    TRAIN_SPLIT_FILE
)

validation_df = pd.read_csv(
    VALIDATION_SPLIT_FILE
)

test_df = pd.read_csv(
    TEST_SPLIT_FILE
)


for df in [
    genomic_df,
    full_df,
    train_df,
    validation_df,
    test_df
]:

    df["condition_id"] = (
        df["condition_id"]
        .astype(str)
        .str.strip()
    )


# =============================================================================
# 3. FROZEN SPLIT GATE
# =============================================================================

full_ids = set(
    full_df["condition_id"]
)

train_ids = set(
    train_df["condition_id"]
)

validation_ids = set(
    validation_df["condition_id"]
)

test_ids = set(
    test_df["condition_id"]
)

print(
    "Full conditions:",
    len(full_ids)
)

print(
    "Train conditions:",
    len(train_ids)
)

print(
    "Validation conditions:",
    len(validation_ids)
)

print(
    "Test conditions:",
    len(test_ids)
)

if len(full_ids) != 1976:
    raise RuntimeError(
        "Full cohort must contain 1976 conditions."
    )

if len(train_ids) != 1185:
    raise RuntimeError(
        "Train split must contain 1185 conditions."
    )

if len(validation_ids) != 395:
    raise RuntimeError(
        "Validation split must contain 395 conditions."
    )

if len(test_ids) != 396:
    raise RuntimeError(
        "Test split must contain 396 conditions."
    )

if train_ids & validation_ids:
    raise RuntimeError(
        "Train/validation overlap detected."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test overlap detected."
    )

if validation_ids & test_ids:
    raise RuntimeError(
        "Validation/test overlap detected."
    )

print(
    "Frozen split integrity: PASS"
)


# =============================================================================
# 4. FINAL GENOMIC DATASET GATE
# =============================================================================

required_columns = [
    "condition_id",
    "target_binary",
    "main_lineage",
    "sub_lineage",
    "gene_snp_mutations_aggregated",
    "mutation_token_count",
]

missing = [
    c
    for c in required_columns
    if c not in genomic_df.columns
]

if missing:

    raise RuntimeError(
        "Required genomic columns missing: "
        f"{missing}"
    )

if len(genomic_df) != 1976:

    raise RuntimeError(
        "Final genomic dataset must contain "
        "exactly 1976 conditions."
    )

if genomic_df["condition_id"].nunique() != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )

if set(genomic_df["condition_id"]) != full_ids:

    raise RuntimeError(
        "Final genomic dataset does not exactly "
        "match the frozen 1976-condition cohort."
    )

print(
    "Final genomic cohort gate: PASS"
)


# =============================================================================
# 5. ASSIGN FROZEN SPLIT
# =============================================================================

split_map = {}

for cid in train_ids:
    split_map[cid] = "train"

for cid in validation_ids:
    split_map[cid] = "validation"

for cid in test_ids:
    split_map[cid] = "test"


genomic_df["split"] = (
    genomic_df["condition_id"]
    .map(split_map)
)

if genomic_df["split"].isna().any():

    raise RuntimeError(
        "Some genomic conditions have no split assignment."
    )

print("\nSPLIT ASSIGNMENT")
print("-" * 80)

print(
    genomic_df["split"]
    .value_counts()
    .to_string()
)


# =============================================================================
# 6. TOKENIZATION FUNCTION
# =============================================================================

def normalize_token(token):

    token = str(
        token
    ).strip()

    if not token:
        return None

    token = re.sub(
        r"\s+",
        "_",
        token
    )

    return token


def tokenize_mutations(text):

    if pd.isna(text):

        return []

    text = str(
        text
    ).strip()

    if not text:

        return []

    raw_tokens = re.split(
        r"\s*\|\s*",
        text
    )

    tokens = []

    for raw in raw_tokens:

        raw = raw.strip()

        if not raw:
            continue

        # Secondary separation for records that may
        # contain comma/semicolon-delimited mutations.
        pieces = re.split(
            r"\s*[;,]\s*",
            raw
        )

        for piece in pieces:

            normalized = normalize_token(
                piece
            )

            if normalized:

                tokens.append(
                    normalized
                )

    return tokens


# =============================================================================
# 7. CREATE RAW GENOMIC TOKENS
# =============================================================================

genomic_df[
    "mutation_tokens"
] = (
    genomic_df[
        "gene_snp_mutations_aggregated"
    ]
    .apply(
        tokenize_mutations
    )
)


# =============================================================================
# 8. ADD LINEAGE TOKENS
# =============================================================================

def lineage_token(value):

    if pd.isna(value):

        return None

    value = str(
        value
    ).strip()

    if not value:

        return None

    return normalize_token(
        value
    )


genomic_df[
    "main_lineage_token"
] = (
    genomic_df[
        "main_lineage"
    ]
    .apply(
        lineage_token
    )
)

genomic_df[
    "sub_lineage_token"
] = (
    genomic_df[
        "sub_lineage"
    ]
    .apply(
        lineage_token
    )
)


# =============================================================================
# 9. TRAINING-ONLY TOKEN COLLECTION
# =============================================================================

training_df = genomic_df[
    genomic_df["split"] == "train"
].copy()

validation_genomic_df = genomic_df[
    genomic_df["split"] == "validation"
].copy()

test_genomic_df = genomic_df[
    genomic_df["split"] == "test"
].copy()


if len(training_df) != 1185:
    raise RuntimeError(
        "Training genomic cohort must contain 1185 conditions."
    )

if len(validation_genomic_df) != 395:
    raise RuntimeError(
        "Validation genomic cohort must contain 395 conditions."
    )

if len(test_genomic_df) != 396:
    raise RuntimeError(
        "Test genomic cohort must contain 396 conditions."
    )


print("\nTRAINING-ONLY VOCABULARY SOURCE")
print("-" * 80)

print(
    "Training conditions:",
    len(training_df)
)

print(
    "Validation conditions:",
    len(validation_genomic_df)
)

print(
    "Test conditions:",
    len(test_genomic_df)
)


# =============================================================================
# 10. BUILD TRAINING-ONLY TOKEN COUNTS
# =============================================================================

token_counter = Counter()

for _, row in training_df.iterrows():

    for token in row[
        "mutation_tokens"
    ]:

        token_counter[
            token
        ] += 1

    main_lineage = row[
        "main_lineage_token"
    ]

    if main_lineage:

        token_counter[
            f"MAIN_LINEAGE={main_lineage}"
        ] += 1

    sub_lineage = row[
        "sub_lineage_token"
    ]

    if sub_lineage:

        token_counter[
            f"SUB_LINEAGE={sub_lineage}"
        ] += 1


# =============================================================================
# 11. VOCABULARY SIZE
# =============================================================================

print("\nTRAINING VOCABULARY")
print("-" * 80)

print(
    "Unique training tokens:",
    len(token_counter)
)

if len(token_counter) == 0:

    raise RuntimeError(
        "Training vocabulary is empty."
    )


# =============================================================================
# 12. SPECIAL TOKENS
# =============================================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
CLS_TOKEN = "<CLS>"
SEP_TOKEN = "<SEP>"

special_tokens = [
    PAD_TOKEN,
    UNK_TOKEN,
    CLS_TOKEN,
    SEP_TOKEN,
]


# =============================================================================
# 13. CREATE VOCABULARY
# =============================================================================

# Deterministic ordering:
# frequency descending, then lexical order.

sorted_tokens = sorted(
    token_counter.items(),
    key=lambda x: (
        -x[1],
        x[0]
    )
)

vocab = {}

for token in special_tokens:

    vocab[token] = len(vocab)


for token, count in sorted_tokens:

    if token in vocab:
        continue

    vocab[token] = len(vocab)


print(
    "Final vocabulary size:",
    len(vocab)
)

print(
    "Special tokens:",
    special_tokens
)


# =============================================================================
# 14. TRAINING-ONLY VOCABULARY LEAKAGE CHECK
# =============================================================================

train_token_set = set(
    token_counter.keys()
)

validation_token_set = set()

for _, row in validation_genomic_df.iterrows():

    validation_token_set.update(
        row[
            "mutation_tokens"
        ]
    )

    if row[
        "main_lineage_token"
    ]:

        validation_token_set.add(
            f"MAIN_LINEAGE={row['main_lineage_token']}"
        )

    if row[
        "sub_lineage_token"
    ]:

        validation_token_set.add(
            f"SUB_LINEAGE={row['sub_lineage_token']}"
        )


test_token_set = set()

for _, row in test_genomic_df.iterrows():

    test_token_set.update(
        row[
            "mutation_tokens"
        ]
    )

    if row[
        "main_lineage_token"
    ]:

        test_token_set.add(
            f"MAIN_LINEAGE={row['main_lineage_token']}"
        )

    if row[
        "sub_lineage_token"
    ]:

        test_token_set.add(
            f"SUB_LINEAGE={row['sub_lineage_token']}"
        )


validation_only_tokens = (
    validation_token_set
    - train_token_set
)

test_only_tokens = (
    test_token_set
    - train_token_set
)

print("\nVOCABULARY LEAKAGE AUDIT")
print("-" * 80)

print(
    "Validation tokens absent from training:",
    len(validation_only_tokens)
)

print(
    "Test tokens absent from training:",
    len(test_only_tokens)
)

print(
    "These tokens will map to <UNK>."
)

print(
    "Vocabulary construction used validation/test:",
    "NO"
)


# =============================================================================
# 15. BUILD SEQUENCES
# =============================================================================

def row_to_tokens(row):

    tokens = []

    tokens.append(
        CLS_TOKEN
    )

    # Main lineage
    if row[
        "main_lineage_token"
    ]:

        tokens.append(
            f"MAIN_LINEAGE={row['main_lineage_token']}"
        )

    # Sub-lineage
    if row[
        "sub_lineage_token"
    ]:

        tokens.append(
            f"SUB_LINEAGE={row['sub_lineage_token']}"
        )

    # Mutation tokens
    for token in row[
        "mutation_tokens"
    ]:

        tokens.append(
            token
        )

    tokens.append(
        SEP_TOKEN
    )

    return tokens


genomic_df[
    "raw_sequence_tokens"
] = genomic_df.apply(
    row_to_tokens,
    axis=1
)


# =============================================================================
# 16. CONVERT TOKENS TO IDS
# =============================================================================

def tokens_to_ids(tokens):

    return [
        vocab.get(
            token,
            vocab[UNK_TOKEN]
        )
        for token in tokens
    ]


genomic_df[
    "token_ids"
] = genomic_df[
    "raw_sequence_tokens"
].apply(
    tokens_to_ids
)


# =============================================================================
# 17. SEQUENCE LENGTH AUDIT
# =============================================================================

genomic_df[
    "sequence_length"
] = genomic_df[
    "token_ids"
].apply(
    len
)

print("\nSEQUENCE LENGTH AUDIT")
print("-" * 80)

print(
    "Minimum:",
    int(
        genomic_df[
            "sequence_length"
        ].min()
    )
)

print(
    "Maximum:",
    int(
        genomic_df[
            "sequence_length"
        ].max()
    )
)

print(
    "Mean:",
    float(
        genomic_df[
            "sequence_length"
        ].mean()
    )
)

print(
    "Median:",
    float(
        genomic_df[
            "sequence_length"
        ].median()
    )
)

print(
    "95th percentile:",
    float(
        genomic_df[
            "sequence_length"
        ].quantile(
            0.95
        )
    )
)

print(
    "99th percentile:",
    float(
        genomic_df[
            "sequence_length"
        ].quantile(
            0.99
        )
    )
)


# =============================================================================
# 18. MAX SEQUENCE LENGTH
# =============================================================================

MAX_SEQ_LEN = 74

print(
    "\nConfigured MAX_SEQ_LEN:",
    MAX_SEQ_LEN
)

truncated_count = int(
    (
        genomic_df[
            "sequence_length"
        ]
        > MAX_SEQ_LEN
    ).sum()
)

print(
    "Sequences longer than MAX_SEQ_LEN:",
    truncated_count
)

if truncated_count > 0:

    print(
        "WARNING: sequences will require "
        "deterministic truncation."
    )


# =============================================================================
# 19. PAD/TRUNCATE
# =============================================================================

PAD_ID = vocab[
    PAD_TOKEN
]

UNK_ID = vocab[
    UNK_TOKEN
]


def pad_or_truncate(
    ids,
    max_len=MAX_SEQ_LEN
):

    ids = list(
        ids
    )

    if len(ids) > max_len:

        # Preserve CLS and SEP.
        ids = (
            ids[:max_len - 1]
            + [vocab[SEP_TOKEN]]
        )

    elif len(ids) < max_len:

        ids = (
            ids
            +
            [PAD_ID]
            *
            (
                max_len
                - len(ids)
            )
        )

    return ids


genomic_df[
    "input_ids"
] = genomic_df[
    "token_ids"
].apply(
    pad_or_truncate
)


# =============================================================================
# 20. ATTENTION MASK
# =============================================================================

genomic_df[
    "attention_mask"
] = genomic_df[
    "input_ids"
].apply(
    lambda ids: [
        0 if token == PAD_ID else 1
        for token in ids
    ]
)


# =============================================================================
# 21. FINAL SEQUENCE SHAPE CHECK
# =============================================================================

sequence_lengths = genomic_df[
    "input_ids"
].apply(
    len
)

mask_lengths = genomic_df[
    "attention_mask"
].apply(
    len
)

if not (
    sequence_lengths
    == MAX_SEQ_LEN
).all():

    raise RuntimeError(
        "Not all input sequences have "
        "the required length."
    )

if not (
    mask_lengths
    == MAX_SEQ_LEN
).all():

    raise RuntimeError(
        "Attention masks have incorrect length."
    )

print(
    "\nFinal sequence shape:",
    f"[{len(genomic_df)}, {MAX_SEQ_LEN}]"
)

print(
    "Sequence shape gate: PASS"
)


# =============================================================================
# 22. UNK USAGE AUDIT
# =============================================================================

genomic_df[
    "unk_count"
] = genomic_df[
    "input_ids"
].apply(
    lambda ids: int(
        sum(
            token == UNK_ID
            for token in ids
        )
    )
)

print("\n<UNK> AUDIT")
print("-" * 80)

print(
    "Conditions containing <UNK>:",
    int(
        (
            genomic_df[
                "unk_count"
            ] > 0
        ).sum()
    )
)

print(
    "Total <UNK> tokens:",
    int(
        genomic_df[
            "unk_count"
        ].sum()
    )
)


# =============================================================================
# 23. SPLIT-SPECIFIC SEQUENCE AUDIT
# =============================================================================

print("\nSPLIT-SPECIFIC SEQUENCE AUDIT")
print("-" * 80)

for split_name in [
    "train",
    "validation",
    "test"
]:

    subset = genomic_df[
        genomic_df["split"]
        == split_name
    ]

    print(
        f"{split_name:<12}:",
        len(subset),
        "conditions"
    )

    print(
        f"{'':12} DS=",
        int(
            (
                subset[
                    "target_binary"
                ] == 0
            ).sum()
        ),
        "DR=",
        int(
            (
                subset[
                    "target_binary"
                ] == 1
            ).sum()
        )
    )


# =============================================================================
# 24. FINAL LEAKAGE GATE
# =============================================================================

print("\nFINAL LEAKAGE GATE")
print("-" * 80)

# Vocabulary was built only from train.
vocab_source = "TRAIN_ONLY"

if vocab_source != "TRAIN_ONLY":

    raise RuntimeError(
        "Vocabulary source is not training-only."
    )

# Confirm no split overlap.
if train_ids & validation_ids:
    raise RuntimeError(
        "Train/validation overlap."
    )

if train_ids & test_ids:
    raise RuntimeError(
        "Train/test overlap."
    )

if validation_ids & test_ids:
    raise RuntimeError(
        "Validation/test overlap."
    )

# Confirm target-derived fields are not present.
for forbidden in [
    "drug_resistance_type",
    "type_of_resistance",
    "num_drug_resistant_variants",
]:

    if forbidden in genomic_df.columns:

        raise RuntimeError(
            f"Forbidden target-derived field found: "
            f"{forbidden}"
        )

print(
    "Vocabulary source: TRAIN ONLY"
)

print(
    "Validation used for vocabulary: NO"
)

print(
    "Test used for vocabulary: NO"
)

print(
    "Target-derived resistance fields: EXCLUDED"
)

print(
    "Leakage gate: PASS"
)


# =============================================================================
# 25. SAVE VOCABULARY
# =============================================================================

VOCAB_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_ONLY_Vocabulary.json"
)

with open(
    VOCAB_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "special_tokens":
                special_tokens,

            "pad_token":
                PAD_TOKEN,

            "unk_token":
                UNK_TOKEN,

            "cls_token":
                CLS_TOKEN,

            "sep_token":
                SEP_TOKEN,

            "pad_id":
                PAD_ID,

            "unk_id":
                UNK_ID,

            "cls_id":
                vocab[CLS_TOKEN],

            "sep_id":
                vocab[SEP_TOKEN],

            "vocab_size":
                len(vocab),

            "max_seq_len":
                MAX_SEQ_LEN,

            "vocabulary_source":
                "TRAIN_ONLY",

            "tokens":
                vocab,

            "token_counts":
                dict(token_counter),
        },
        f,
        indent=2
    )


# =============================================================================
# 26. SAVE FINAL SEQUENCE DATASET
# =============================================================================

SEQUENCE_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

save_columns = [
    "condition_id",
    "target_binary",
    "split",
    "main_lineage",
    "sub_lineage",
    "gene_snp_mutations_aggregated",
    "mutation_token_count",
    "has_mutation_evidence",
    "has_lineage_evidence",
    "sequence_length",
    "unk_count",
    "input_ids",
    "attention_mask",
]

genomic_df[
    save_columns
].to_csv(
    SEQUENCE_FILE,
    index=False
)


# =============================================================================
# 27. SAVE NUMPY ARRAYS
# =============================================================================

input_ids_array = np.asarray(
    genomic_df[
        "input_ids"
    ].tolist(),
    dtype=np.int64
)

attention_mask_array = np.asarray(
    genomic_df[
        "attention_mask"
    ].tolist(),
    dtype=np.int64
)

targets_array = genomic_df[
    "target_binary"
].to_numpy(
    dtype=np.int64
)

condition_ids_array = genomic_df[
    "condition_id"
].to_numpy(
    dtype=str
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_Input_IDs.npy",
    input_ids_array
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_Attention_Masks.npy",
    attention_mask_array
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_Targets.npy",
    targets_array
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_Condition_IDs.npy",
    condition_ids_array
)


# =============================================================================
# 28. SAVE SPLIT INDICES
# =============================================================================

condition_to_index = {
    cid: idx
    for idx, cid in enumerate(
        genomic_df[
            "condition_id"
        ]
    )
}

train_indices = np.asarray(
    [
        condition_to_index[cid]
        for cid in train_ids
    ],
    dtype=np.int64
)

validation_indices = np.asarray(
    [
        condition_to_index[cid]
        for cid in validation_ids
    ],
    dtype=np.int64
)

test_indices = np.asarray(
    [
        condition_to_index[cid]
        for cid in test_ids
    ],
    dtype=np.int64
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_Indices.npy",
    train_indices
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_VALIDATION_Indices.npy",
    validation_indices
)

np.save(
    CELL5_ROOT
    / "Notebook13_Cell5_TEST_Indices.npy",
    test_indices
)


# =============================================================================
# 29. SHA256 HELPER
# =============================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            h.update(chunk)

    return h.hexdigest()


# =============================================================================
# 30. SUMMARY
# =============================================================================

summary = {

    "notebook":
        "13_Final_1976_Genomic_Preparation",

    "cell":
        "Cell5_Training_Only_Vocabulary",

    "final_conditions":
        1976,

    "train_conditions":
        1185,

    "validation_conditions":
        395,

    "test_conditions":
        396,

    "vocab_size":
        len(vocab),

    "max_sequence_length":
        MAX_SEQ_LEN,

    "training_unique_tokens":
        len(token_counter),

    "validation_only_tokens":
        len(validation_only_tokens),

    "test_only_tokens":
        len(test_only_tokens),

    "sequences_longer_than_max":
        truncated_count,

    "conditions_with_unk":
        int(
            (
                genomic_df[
                    "unk_count"
                ] > 0
            ).sum()
        ),

    "total_unk_tokens":
        int(
            genomic_df[
                "unk_count"
            ].sum()
        ),

    "vocabulary_source":
        "TRAIN_ONLY",

    "validation_used_for_vocab":
        False,

    "test_used_for_vocab":
        False,

    "target_derived_fields_excluded":
        True,

    "training_performed":
        False,

    "test_predictions":
        False,

    "status":
        "PASS_TRAINING_ONLY_SEQUENCE_PREPARATION"

}

SUMMARY_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 31. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 5 STATUS: PASS")
print("=" * 80)

print(
    "Final conditions:",
    1976
)

print(
    "Training conditions:",
    1185
)

print(
    "Validation conditions:",
    395
)

print(
    "Test conditions:",
    396
)

print(
    "Training-only vocabulary size:",
    len(vocab)
)

print(
    "MAX_SEQ_LEN:",
    MAX_SEQ_LEN
)

print(
    "Vocabulary leakage:",
    "NONE"
)

print(
    "Target-derived resistance fields:",
    "EXCLUDED"
)

print(
    "Training:",
    "NOT PERFORMED"
)

print(
    "Test predictions:",
    "NOT PERFORMED"
)

print(
    "\nVocabulary:"
)

print(
    VOCAB_FILE
)

print(
    "\nSequence dataset:"
)

print(
    SEQUENCE_FILE
)

print(
    "\nSummary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "The genomic Transformer should NOT be trained "
    "until Cell 5 output is reviewed."
)

NOTEBOOK 13 — CELL 5
TRAINING-ONLY GENOMIC VOCABULARY AND SEQUENCE PREPARATION

DATA LOADING
--------------------------------------------------------------------------------
Full conditions: 1976
Train conditions: 1185
Validation conditions: 395
Test conditions: 396
Frozen split integrity: PASS
Final genomic cohort gate: PASS

SPLIT ASSIGNMENT
--------------------------------------------------------------------------------
split
train         1185
test           396
validation     395

TRAINING-ONLY VOCABULARY SOURCE
--------------------------------------------------------------------------------
Training conditions: 1185
Validation conditions: 395
Test conditions: 396

TRAINING VOCABULARY
--------------------------------------------------------------------------------
Unique training tokens: 74
Final vocabulary size: 78
Special tokens: ['<PAD>', '<UNK>', '<CLS>', '<SEP>']

VOCABULARY LEAKAGE AUDIT
--------------------------------------------------------------------------------
Validat

In [8]:
# =============================================================================
# NOTEBOOK 13 — CELL 6
# FINAL GENOMIC TRANSFORMER
# DATASET / DATALOADER / MODEL ARCHITECTURE AUDIT
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


print("=" * 80)
print("NOTEBOOK 13 — CELL 6")
print("FINAL GENOMIC TRANSFORMER ARCHITECTURE AUDIT")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

CELL5_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell5_Training_Only_Vocabulary"
)

CELL6_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell6_Transformer_Architecture"
)

CELL6_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SEQUENCE_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

VOCAB_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_ONLY_Vocabulary.json"
)


# =============================================================================
# 3. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

if not SEQUENCE_FILE.exists():
    raise FileNotFoundError(
        f"Sequence file not found:\n{SEQUENCE_FILE}"
    )

if not VOCAB_FILE.exists():
    raise FileNotFoundError(
        f"Vocabulary file not found:\n{VOCAB_FILE}"
    )

print(
    "Sequence dataset : PASS"
)

print(
    "Vocabulary       : PASS"
)


# =============================================================================
# 4. LOAD DATA
# =============================================================================

sequence_df = pd.read_csv(
    SEQUENCE_FILE,
    low_memory=False
)

with open(
    VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    vocab_data = json.load(f)


print("\nDATA LOADING")
print("-" * 80)

print(
    "Rows:",
    len(sequence_df)
)

print(
    "Vocabulary size:",
    vocab_data["vocab_size"]
)


# =============================================================================
# 5. FINAL DATASET GATE
# =============================================================================

if len(sequence_df) != 1976:
    raise RuntimeError(
        "Expected exactly 1976 genomic conditions."
    )

if sequence_df[
    "condition_id"
].nunique() != 1976:
    raise RuntimeError(
        "Condition IDs are not unique."
    )

expected_split_counts = {
    "train": 1185,
    "validation": 395,
    "test": 396,
}

actual_split_counts = (
    sequence_df[
        "split"
    ]
    .value_counts()
    .to_dict()
)

for split_name, expected in (
    expected_split_counts.items()
):

    actual = actual_split_counts.get(
        split_name,
        0
    )

    if actual != expected:
        raise RuntimeError(
            f"{split_name} expected {expected}, "
            f"found {actual}."
        )

print(
    "1976-condition dataset: PASS"
)

print(
    "Split counts: PASS"
)


# =============================================================================
# 6. TARGET GATE
# =============================================================================

target_counts = (
    sequence_df[
        "target_binary"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

print("\nTARGET DISTRIBUTION")
print("-" * 80)

print(
    "DS (0):",
    target_counts.get(0, 0)
)

print(
    "DR (1):",
    target_counts.get(1, 0)
)

if target_counts.get(0, 0) != 704:
    raise RuntimeError(
        "DS count changed."
    )

if target_counts.get(1, 0) != 1272:
    raise RuntimeError(
        "DR count changed."
    )

print(
    "Target distribution: PASS"
)


# =============================================================================
# 7. SEQUENCE CONFIGURATION
# =============================================================================

MAX_SEQ_LEN = int(
    vocab_data[
        "max_seq_len"
    ]
)

VOCAB_SIZE = int(
    vocab_data[
        "vocab_size"
    ]
)

PAD_ID = int(
    vocab_data[
        "pad_id"
    ]
)

print("\nSEQUENCE CONFIGURATION")
print("-" * 80)

print(
    "Vocabulary size:",
    VOCAB_SIZE
)

print(
    "Maximum sequence length:",
    MAX_SEQ_LEN
)

print(
    "PAD ID:",
    PAD_ID
)

if VOCAB_SIZE != 78:
    raise RuntimeError(
        f"Expected vocabulary size 78, "
        f"found {VOCAB_SIZE}."
    )

if MAX_SEQ_LEN != 74:
    raise RuntimeError(
        f"Expected MAX_SEQ_LEN 74, "
        f"found {MAX_SEQ_LEN}."
    )


# =============================================================================
# 8. PARSE STORED ARRAYS
# =============================================================================

def parse_array(value):

    if isinstance(
        value,
        list
    ):
        return value

    value = str(
        value
    ).strip()

    value = (
        value
        .replace("[", "")
        .replace("]", "")
    )

    if not value:
        return []

    return [
        int(x.strip())
        for x in value.split(",")
        if x.strip()
    ]


sequence_df[
    "input_ids_parsed"
] = sequence_df[
    "input_ids"
].apply(
    parse_array
)

sequence_df[
    "attention_mask_parsed"
] = sequence_df[
    "attention_mask"
].apply(
    parse_array
)


# =============================================================================
# 9. SEQUENCE SHAPE VALIDATION
# =============================================================================

input_lengths = (
    sequence_df[
        "input_ids_parsed"
    ]
    .apply(len)
)

mask_lengths = (
    sequence_df[
        "attention_mask_parsed"
    ]
    .apply(len)
)

if not (
    input_lengths == MAX_SEQ_LEN
).all():

    raise RuntimeError(
        "Invalid input sequence length detected."
    )

if not (
    mask_lengths == MAX_SEQ_LEN
).all():

    raise RuntimeError(
        "Invalid attention-mask length detected."
    )

print("\nSEQUENCE SHAPE")
print("-" * 80)

print(
    "Input shape:",
    f"(1976, {MAX_SEQ_LEN})"
)

print(
    "Attention-mask shape:",
    f"(1976, {MAX_SEQ_LEN})"
)

print(
    "Sequence shape: PASS"
)


# =============================================================================
# 10. TOKEN ID RANGE CHECK
# =============================================================================

all_input_ids = np.concatenate(
    [
        np.asarray(
            x,
            dtype=np.int64
        )
        for x in sequence_df[
            "input_ids_parsed"
        ]
    ]
)

if all_input_ids.min() < 0:
    raise RuntimeError(
        "Negative token ID detected."
    )

if all_input_ids.max() >= VOCAB_SIZE:
    raise RuntimeError(
        "Token ID exceeds vocabulary size."
    )

print(
    "Token ID range: PASS"
)


# =============================================================================
# 11. ATTENTION MASK CHECK
# =============================================================================

all_masks = np.concatenate(
    [
        np.asarray(
            x,
            dtype=np.int64
        )
        for x in sequence_df[
            "attention_mask_parsed"
        ]
    ]
)

unique_mask_values = set(
    np.unique(
        all_masks
    ).tolist()
)

if not unique_mask_values.issubset(
    {0, 1}
):

    raise RuntimeError(
        "Attention mask contains values "
        "other than 0 and 1."
    )

print(
    "Attention mask values: PASS"
)


# =============================================================================
# 12. DATASET CLASS
# =============================================================================

class GenomicDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.input_ids = np.asarray(
            self.df[
                "input_ids_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.attention_masks = np.asarray(
            self.df[
                "attention_mask_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.targets = (
            self.df[
                "target_binary"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        self.condition_ids = (
            self.df[
                "condition_id"
            ]
            .astype(str)
            .to_numpy()
        )

    def __len__(self):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        index
    ):

        return {

            "input_ids":
                torch.tensor(
                    self.input_ids[index],
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    self.attention_masks[index],
                    dtype=torch.long
                ),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.long
                ),

            "condition_id":
                self.condition_ids[index],
        }


# =============================================================================
# 13. SPLIT DATASETS
# =============================================================================

train_data = sequence_df[
    sequence_df["split"] == "train"
].copy()

validation_data = sequence_df[
    sequence_df["split"] == "validation"
].copy()

test_data = sequence_df[
    sequence_df["split"] == "test"
].copy()


train_dataset = GenomicDataset(
    train_data
)

validation_dataset = GenomicDataset(
    validation_data
)

test_dataset = GenomicDataset(
    test_data
)


print("\nDATASET OBJECTS")
print("-" * 80)

print(
    "Train:",
    len(train_dataset)
)

print(
    "Validation:",
    len(validation_dataset)
)

print(
    "Test:",
    len(test_dataset)
)


# =============================================================================
# 14. DATALOADERS
# =============================================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Test loader is created only for structural validation.
# It will NOT be used during model training or checkpoint selection.

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


print("\nDATALOADER CONFIGURATION")
print("-" * 80)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(validation_loader)
)

print(
    "Test batches:",
    len(test_loader)
)


# =============================================================================
# 15. TRANSFORMER CONFIGURATION
# =============================================================================

EMBED_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.20
NUM_CLASSES = 2


if EMBED_DIM % NUM_HEADS != 0:

    raise RuntimeError(
        "Embedding dimension must be divisible "
        "by number of attention heads."
    )


# =============================================================================
# 16. FINAL GENOMIC TRANSFORMER
# =============================================================================

class GenomicTransformer(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.20,
        max_seq_len=74,
        num_classes=2,
        pad_id=0,
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            embedding_dim=embed_dim
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=False
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                embed_dim
            ),
            nn.Linear(
                embed_dim,
                num_classes
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        return_embedding=False
    ):

        batch_size, seq_len = (
            input_ids.shape
        )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        positions = (
            positions
            .unsqueeze(0)
            .expand(
                batch_size,
                seq_len
            )
        )

        x = (
            self.token_embedding(
                input_ids
            )
            +
            self.position_embedding(
                positions
            )
        )

        x = self.dropout(
            x
        )

        padding_mask = (
            attention_mask == 0
        )

        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        # CLS token representation.
        cls_embedding = x[:, 0, :]

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        if return_embedding:

            return (
                logits,
                cls_embedding
            )

        return logits


# =============================================================================
# 17. DEVICE
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)


# =============================================================================
# 18. MODEL INSTANTIATION
# =============================================================================

model = GenomicTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
    max_seq_len=MAX_SEQ_LEN,
    num_classes=NUM_CLASSES,
    pad_id=PAD_ID
).to(device)


# =============================================================================
# 19. PARAMETER COUNT
# =============================================================================

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print("\nMODEL ARCHITECTURE")
print("-" * 80)

print(
    "Vocabulary size:",
    VOCAB_SIZE
)

print(
    "Embedding dimension:",
    EMBED_DIM
)

print(
    "Attention heads:",
    NUM_HEADS
)

print(
    "Transformer layers:",
    NUM_LAYERS
)

print(
    "Feed-forward dimension:",
    FF_DIM
)

print(
    "Dropout:",
    DROPOUT
)

print(
    "Maximum sequence length:",
    MAX_SEQ_LEN
)

print(
    "Number of classes:",
    NUM_CLASSES
)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)


# =============================================================================
# 20. FORWARD-PASS SANITY CHECK
# =============================================================================

batch = next(
    iter(
        train_loader
    )
)

batch_input_ids = batch[
    "input_ids"
].to(device)

batch_attention_mask = batch[
    "attention_mask"
].to(device)

batch_targets = batch[
    "target"
].to(device)


model.eval()

with torch.no_grad():

    logits, embeddings = model(
        batch_input_ids,
        batch_attention_mask,
        return_embedding=True
    )


print("\nFORWARD PASS")
print("-" * 80)

print(
    "Input shape:",
    tuple(
        batch_input_ids.shape
    )
)

print(
    "Attention mask shape:",
    tuple(
        batch_attention_mask.shape
    )
)

print(
    "Target shape:",
    tuple(
        batch_targets.shape
    )
)

print(
    "Logits shape:",
    tuple(
        logits.shape
    )
)

print(
    "Embedding shape:",
    tuple(
        embeddings.shape
    )
)


if logits.shape != (
    BATCH_SIZE,
    NUM_CLASSES
):

    raise RuntimeError(
        "Unexpected logits shape."
    )

if embeddings.shape != (
    BATCH_SIZE,
    EMBED_DIM
):

    raise RuntimeError(
        "Unexpected genomic embedding shape."
    )

print(
    "Forward pass: PASS"
)


# =============================================================================
# 21. FINITE-VALUE CHECK
# =============================================================================

if not torch.isfinite(
    logits
).all():

    raise RuntimeError(
        "Non-finite logits detected."
    )

if not torch.isfinite(
    embeddings
).all():

    raise RuntimeError(
        "Non-finite embeddings detected."
    )

print(
    "Finite-value check: PASS"
)


# =============================================================================
# 22. TEST ISOLATION
# =============================================================================

print("\nTEST ISOLATION")
print("-" * 80)

print(
    "Test dataset created for structural validation only."
)

print(
    "Test used for training:",
    "NO"
)

print(
    "Test used for checkpoint selection:",
    "NO"
)

print(
    "Test used for threshold tuning:",
    "NO"
)

print(
    "Test predictions generated:",
    "NO"
)


# =============================================================================
# 23. SAVE ARCHITECTURE CONFIGURATION
# =============================================================================

architecture_config = {

    "seed":
        SEED,

    "vocab_size":
        VOCAB_SIZE,

    "max_seq_len":
        MAX_SEQ_LEN,

    "embedding_dimension":
        EMBED_DIM,

    "attention_heads":
        NUM_HEADS,

    "transformer_layers":
        NUM_LAYERS,

    "feed_forward_dimension":
        FF_DIM,

    "dropout":
        DROPOUT,

    "num_classes":
        NUM_CLASSES,

    "padding_id":
        PAD_ID,

    "batch_size":
        BATCH_SIZE,

    "total_parameters":
        int(total_parameters),

    "trainable_parameters":
        int(trainable_parameters),

    "device":
        str(device),

    "test_used_for_training":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_tuning":
        False,

    "test_predictions_generated":
        False
}


CONFIG_FILE = (
    CELL6_ROOT
    / "Notebook13_Cell6_Transformer_Architecture.json"
)

with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        architecture_config,
        f,
        indent=4
    )


# =============================================================================
# 24. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 6 STATUS: PASS")
print("=" * 80)

print(
    "Dataset:",
    "PASS"
)

print(
    "Dataloaders:",
    "PASS"
)

print(
    "Transformer architecture:",
    "PASS"
)

print(
    "Forward pass:",
    "PASS"
)

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)

print(
    "\nNo training performed."
)

print(
    "No test predictions performed."
)

print(
    "\nArchitecture configuration:"
)

print(
    CONFIG_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "The fixed-epoch training cell will be the next cell."
)

NOTEBOOK 13 — CELL 6
FINAL GENOMIC TRANSFORMER ARCHITECTURE AUDIT

PATH GATE
--------------------------------------------------------------------------------
Sequence dataset : PASS
Vocabulary       : PASS

DATA LOADING
--------------------------------------------------------------------------------
Rows: 1976
Vocabulary size: 78
1976-condition dataset: PASS
Split counts: PASS

TARGET DISTRIBUTION
--------------------------------------------------------------------------------
DS (0): 704
DR (1): 1272
Target distribution: PASS

SEQUENCE CONFIGURATION
--------------------------------------------------------------------------------
Vocabulary size: 78
Maximum sequence length: 74
PAD ID: 0

SEQUENCE SHAPE
--------------------------------------------------------------------------------
Input shape: (1976, 74)
Attention-mask shape: (1976, 74)
Sequence shape: PASS
Token ID range: PASS
Attention mask values: PASS

DATASET OBJECTS
---------------------------------------------------------------

C:\Users\Gobika\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\nn\modules\transformer.py:529: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


In [9]:
# =============================================================================
# NOTEBOOK 13 — CELL 7
# FINAL GENOMIC TRANSFORMER TRAINING
# FIXED-EPOCH TRAINING — NO EARLY STOPPING
# =============================================================================

from pathlib import Path
import json
import random
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


print("=" * 80)
print("NOTEBOOK 13 — CELL 7")
print("FINAL GENOMIC TRANSFORMER TRAINING")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

CELL5_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell5_Training_Only_Vocabulary"
)

CELL6_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell6_Transformer_Architecture"
)

CELL7_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell7_Transformer_Training"
)

CELL7_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SEQUENCE_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

VOCAB_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_ONLY_Vocabulary.json"
)

ARCHITECTURE_FILE = (
    CELL6_ROOT
    / "Notebook13_Cell6_Transformer_Architecture.json"
)


# =============================================================================
# 3. TRAINING CONFIGURATION
# =============================================================================

EPOCHS = 40

BATCH_SIZE = 32

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

LABEL_SMOOTHING = 0.02

DROPOUT = 0.20

GRADIENT_CLIP = 1.0

NUM_WORKERS = 0

# No early stopping.
# Best checkpoint is determined only from validation ROC-AUC.
SELECTION_METRIC = "validation_roc_auc"


# =============================================================================
# 4. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

for name, path in {
    "Sequence dataset": SEQUENCE_FILE,
    "Vocabulary": VOCAB_FILE,
    "Architecture": ARCHITECTURE_FILE,
}.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<20}: PASS"
    )


# =============================================================================
# 5. LOAD DATA
# =============================================================================

sequence_df = pd.read_csv(
    SEQUENCE_FILE,
    low_memory=False
)

with open(
    VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    vocab_data = json.load(f)

with open(
    ARCHITECTURE_FILE,
    "r",
    encoding="utf-8"
) as f:

    architecture_data = json.load(f)


# =============================================================================
# 6. FROZEN COHORT GATE
# =============================================================================

if len(sequence_df) != 1976:

    raise RuntimeError(
        "Expected exactly 1976 conditions."
    )

if sequence_df[
    "condition_id"
].nunique() != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )

expected_counts = {
    "train": 1185,
    "validation": 395,
    "test": 396,
}

actual_counts = (
    sequence_df[
        "split"
    ]
    .value_counts()
    .to_dict()
)

for split_name, expected in expected_counts.items():

    actual = actual_counts.get(
        split_name,
        0
    )

    if actual != expected:

        raise RuntimeError(
            f"{split_name}: expected "
            f"{expected}, found {actual}"
        )


print("\nFROZEN DATASET")
print("-" * 80)

print(
    "Full:",
    len(sequence_df)
)

print(
    "Train:",
    actual_counts["train"]
)

print(
    "Validation:",
    actual_counts["validation"]
)

print(
    "Test:",
    actual_counts["test"]
)

print(
    "Frozen cohort: PASS"
)


# =============================================================================
# 7. IMPORTANT TEST-ISOLATION GATE
# =============================================================================

# The test split is intentionally NOT converted into a Dataset
# and will NOT be loaded by the training loop.

train_df = sequence_df[
    sequence_df["split"] == "train"
].copy()

validation_df = sequence_df[
    sequence_df["split"] == "validation"
].copy()


if len(train_df) != 1185:
    raise RuntimeError(
        "Training cohort must contain 1185 conditions."
    )

if len(validation_df) != 395:
    raise RuntimeError(
        "Validation cohort must contain 395 conditions."
    )


# =============================================================================
# 8. TARGET DISTRIBUTION
# =============================================================================

print("\nTRAINING TARGET DISTRIBUTION")
print("-" * 80)

train_ds = int(
    (
        train_df[
            "target_binary"
        ] == 0
    ).sum()
)

train_dr = int(
    (
        train_df[
            "target_binary"
        ] == 1
    ).sum()
)

validation_ds = int(
    (
        validation_df[
            "target_binary"
        ] == 0
    ).sum()
)

validation_dr = int(
    (
        validation_df[
            "target_binary"
        ] == 1
    ).sum()
)

print(
    "Train DS:",
    train_ds
)

print(
    "Train DR:",
    train_dr
)

print(
    "Validation DS:",
    validation_ds
)

print(
    "Validation DR:",
    validation_dr
)

if train_ds != 422 or train_dr != 763:

    raise RuntimeError(
        "Training target distribution changed."
    )

if validation_ds != 141 or validation_dr != 254:

    raise RuntimeError(
        "Validation target distribution changed."
    )


# =============================================================================
# 9. PARSE ARRAYS
# =============================================================================

def parse_array(value):

    value = str(
        value
    ).strip()

    value = (
        value
        .replace("[", "")
        .replace("]", "")
    )

    if not value:

        return []

    return [
        int(x.strip())
        for x in value.split(",")
        if x.strip()
    ]


for df in [
    train_df,
    validation_df
]:

    df[
        "input_ids_parsed"
    ] = df[
        "input_ids"
    ].apply(
        parse_array
    )

    df[
        "attention_mask_parsed"
    ] = df[
        "attention_mask"
    ].apply(
        parse_array
    )


MAX_SEQ_LEN = int(
    vocab_data[
        "max_seq_len"
    ]
)

VOCAB_SIZE = int(
    vocab_data[
        "vocab_size"
    ]
)

PAD_ID = int(
    vocab_data[
        "pad_id"
    ]
)


# =============================================================================
# 10. DATASET CLASS
# =============================================================================

class GenomicDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.input_ids = np.asarray(
            self.df[
                "input_ids_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.attention_masks = np.asarray(
            self.df[
                "attention_mask_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.targets = (
            self.df[
                "target_binary"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        self.condition_ids = (
            self.df[
                "condition_id"
            ]
            .astype(str)
            .to_numpy()
        )

    def __len__(
        self
    ):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        index
    ):

        return {

            "input_ids":
                torch.tensor(
                    self.input_ids[index],
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    self.attention_masks[index],
                    dtype=torch.long
                ),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.long
                ),

            "condition_id":
                self.condition_ids[index]
        }


train_dataset = GenomicDataset(
    train_df
)

validation_dataset = GenomicDataset(
    validation_df
)


# =============================================================================
# 11. DATALOADERS
# =============================================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False
)


# =============================================================================
# 12. MODEL
# =============================================================================

class GenomicTransformer(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.20,
        max_seq_len=74,
        num_classes=2,
        pad_id=0
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_id
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embed_dim
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=False
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                embed_dim
            ),
            nn.Linear(
                embed_dim,
                num_classes
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask
    ):

        batch_size, seq_len = (
            input_ids.shape
        )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        positions = (
            positions
            .unsqueeze(0)
            .expand(
                batch_size,
                seq_len
            )
        )

        x = (
            self.token_embedding(
                input_ids
            )
            +
            self.position_embedding(
                positions
            )
        )

        x = self.dropout(
            x
        )

        padding_mask = (
            attention_mask == 0
        )

        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        cls_embedding = x[:, 0, :]

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        return logits


# =============================================================================
# 13. DEVICE
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)


# =============================================================================
# 14. CREATE MODEL
# =============================================================================

model = GenomicTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    ff_dim=256,
    dropout=DROPOUT,
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    pad_id=PAD_ID
).to(
    device
)


# =============================================================================
# 15. CLASS WEIGHTS
# =============================================================================

# Class weights are calculated ONLY from training labels.
# Validation/test labels are never used.

class_counts = np.bincount(
    train_df[
        "target_binary"
    ].to_numpy(),
    minlength=2
)

class_weights = (
    len(train_df)
    /
    (
        2.0
        * class_counts
    )
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\nCLASS WEIGHTS")
print("-" * 80)

print(
    "DS weight:",
    float(
        class_weights[0]
    )
)

print(
    "DR weight:",
    float(
        class_weights[1]
    )
)


# =============================================================================
# 16. LOSS FUNCTION
# =============================================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=LABEL_SMOOTHING
)


# =============================================================================
# 17. OPTIMIZER
# =============================================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)


# =============================================================================
# 18. LR SCHEDULER
# =============================================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-6
)


# =============================================================================
# 19. TRAINING FUNCTIONS
# =============================================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device
):

    model.train()

    running_loss = 0.0

    all_targets = []
    all_predictions = []

    for batch in loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        targets = batch[
            "target"
        ].to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            input_ids,
            attention_mask
        )

        loss = criterion(
            logits,
            targets
        )

        if not torch.isfinite(
            loss
        ):

            raise RuntimeError(
                "Non-finite training loss detected."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP
        )

        optimizer.step()

        running_loss += (
            loss.item()
            *
            targets.size(0)
        )

        predictions = (
            torch.argmax(
                logits,
                dim=1
            )
        )

        all_targets.extend(
            targets.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    epoch_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    epoch_accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_targets = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch[
                "input_ids"
            ].to(device)

            attention_mask = batch[
                "attention_mask"
            ].to(device)

            targets = batch[
                "target"
            ].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            loss = criterion(
                logits,
                targets
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )[:, 1]

            predictions = (
                torch.argmax(
                    logits,
                    dim=1
                )
            )

            running_loss += (
                loss.item()
                *
                targets.size(0)
            )

            all_targets.extend(
                targets.cpu()
                .numpy()
                .tolist()
            )

            all_predictions.extend(
                predictions.cpu()
                .numpy()
                .tolist()
            )

            all_probabilities.extend(
                probabilities.cpu()
                .numpy()
                .tolist()
            )

    validation_loss = (
        running_loss
        /
        len(loader.dataset)
    )

    validation_accuracy = accuracy_score(
        all_targets,
        all_predictions
    )

    validation_balanced_accuracy = (
        balanced_accuracy_score(
            all_targets,
            all_predictions
        )
    )

    validation_precision = precision_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    validation_sensitivity = recall_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    validation_f1 = f1_score(
        all_targets,
        all_predictions,
        zero_division=0
    )

    validation_roc_auc = roc_auc_score(
        all_targets,
        all_probabilities
    )

    validation_pr_auc = (
        average_precision_score(
            all_targets,
            all_probabilities
        )
    )

    return {

        "loss":
            validation_loss,

        "accuracy":
            validation_accuracy,

        "balanced_accuracy":
            validation_balanced_accuracy,

        "precision":
            validation_precision,

        "sensitivity":
            validation_sensitivity,

        "f1":
            validation_f1,

        "roc_auc":
            validation_roc_auc,

        "pr_auc":
            validation_pr_auc,
    }


# =============================================================================
# 20. TRAINING LOOP
# =============================================================================

print("\n")
print("=" * 80)
print("STARTING FIXED-EPOCH TRAINING")
print("=" * 80)

print(
    "Epochs:",
    EPOCHS
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Checkpoint selection:",
    "VALIDATION ROC-AUC"
)

print(
    "Test evaluation:",
    "NOT PERFORMED"
)


history = []

best_val_auc = -np.inf

best_epoch = None

best_state_dict = None

training_start_time = time.time()


for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start_time = time.time()

    train_loss, train_accuracy = (
        train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device
        )
    )

    validation_metrics = (
        validate_one_epoch(
            model,
            validation_loader,
            criterion,
            device
        )
    )

    scheduler.step()

    current_lr = (
        optimizer.param_groups[0]["lr"]
    )

    epoch_time = (
        time.time()
        -
        epoch_start_time
    )

    record = {

        "epoch":
            epoch,

        "train_loss":
            train_loss,

        "train_accuracy":
            train_accuracy,

        "validation_loss":
            validation_metrics[
                "loss"
            ],

        "validation_accuracy":
            validation_metrics[
                "accuracy"
            ],

        "validation_balanced_accuracy":
            validation_metrics[
                "balanced_accuracy"
            ],

        "validation_precision":
            validation_metrics[
                "precision"
            ],

        "validation_sensitivity":
            validation_metrics[
                "sensitivity"
            ],

        "validation_f1":
            validation_metrics[
                "f1"
            ],

        "validation_roc_auc":
            validation_metrics[
                "roc_auc"
            ],

        "validation_pr_auc":
            validation_metrics[
                "pr_auc"
            ],

        "learning_rate":
            current_lr,

        "epoch_time_seconds":
            epoch_time,
    }

    history.append(
        record
    )

    current_val_auc = (
        validation_metrics[
            "roc_auc"
        ]
    )

    if current_val_auc > best_val_auc:

        best_val_auc = (
            current_val_auc
        )

        best_epoch = epoch

        best_state_dict = copy.deepcopy(
            model.state_dict()
        )

        checkpoint_path = (
            CELL7_ROOT
            / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
        )

        torch.save(
            {
                "epoch":
                    epoch,

                "model_state_dict":
                    best_state_dict,

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "scheduler_state_dict":
                    scheduler.state_dict(),

                "validation_roc_auc":
                    float(
                        current_val_auc
                    ),

                "configuration":
                    {
                        "seed":
                            SEED,

                        "epochs":
                            EPOCHS,

                        "batch_size":
                            BATCH_SIZE,

                        "learning_rate":
                            LEARNING_RATE,

                        "weight_decay":
                            WEIGHT_DECAY,

                        "label_smoothing":
                            LABEL_SMOOTHING,

                        "gradient_clip":
                            GRADIENT_CLIP,

                        "embedding_dimension":
                            128,

                        "attention_heads":
                            4,

                        "transformer_layers":
                            2,

                        "feed_forward_dimension":
                            256,

                        "dropout":
                            DROPOUT,

                        "max_seq_len":
                            MAX_SEQ_LEN,

                        "vocab_size":
                            VOCAB_SIZE,

                    }
            },
            checkpoint_path
        )

        checkpoint_status = " *BEST*"

    else:

        checkpoint_status = ""


    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss {train_loss:.4f} | "
        f"Train Acc {train_accuracy:.4f} | "
        f"Val Loss "
        f"{validation_metrics['loss']:.4f} | "
        f"Val Acc "
        f"{validation_metrics['accuracy']:.4f} | "
        f"Val AUC "
        f"{validation_metrics['roc_auc']:.4f} | "
        f"Val F1 "
        f"{validation_metrics['f1']:.4f} | "
        f"LR "
        f"{current_lr:.2e}"
        f"{checkpoint_status}"
    )


training_time = (
    time.time()
    -
    training_start_time
)


# =============================================================================
# 21. RESTORE BEST VALIDATION CHECKPOINT
# =============================================================================

if best_state_dict is None:

    raise RuntimeError(
        "No validation checkpoint was created."
    )

model.load_state_dict(
    best_state_dict
)

model.eval()


# =============================================================================
# 22. SAVE TRAINING HISTORY
# =============================================================================

history_df = pd.DataFrame(
    history
)

HISTORY_FILE = (
    CELL7_ROOT
    / "Notebook13_Cell7_Training_History.csv"
)

history_df.to_csv(
    HISTORY_FILE,
    index=False
)


# =============================================================================
# 23. SAVE FINAL TRAINING SUMMARY
# =============================================================================

summary = {

    "status":
        "PASS",

    "seed":
        SEED,

    "epochs_completed":
        EPOCHS,

    "early_stopping":
        False,

    "best_epoch":
        int(best_epoch),

    "best_validation_roc_auc":
        float(best_val_auc),

    "best_validation_accuracy":
        float(
            history_df.loc[
                history_df[
                    "epoch"
                ] == best_epoch,
                "validation_accuracy"
            ].iloc[0]
        ),

    "best_validation_f1":
        float(
            history_df.loc[
                history_df[
                    "epoch"
                ] == best_epoch,
                "validation_f1"
            ].iloc[0]
        ),

    "train_conditions":
        1185,

    "validation_conditions":
        395,

    "test_conditions":
        396,

    "test_used_for_training":
        False,

    "test_used_for_checkpoint_selection":
        False,

    "test_used_for_threshold_tuning":
        False,

    "test_predictions_generated":
        False,

    "training_time_seconds":
        float(training_time),

    "checkpoint":
        str(
            CELL7_ROOT
            / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
        ),

    "history_file":
        str(
            HISTORY_FILE
        ),
}


SUMMARY_FILE = (
    CELL7_ROOT
    / "Notebook13_Cell7_Training_Summary.json"
)

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )


# =============================================================================
# 24. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 7 STATUS: PASS")
print("=" * 80)

print(
    "Epochs completed:",
    EPOCHS
)

print(
    "Early stopping:",
    "DISABLED"
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation ROC-AUC:",
    f"{best_val_auc:.6f}"
)

print(
    "Best validation accuracy:",
    f"{summary['best_validation_accuracy']:.6f}"
)

print(
    "Best validation F1:",
    f"{summary['best_validation_f1']:.6f}"
)

print(
    "Test used:",
    "NO"
)

print(
    "Test predictions:",
    "NO"
)

print(
    "\nBest checkpoint:"
)

print(
    CELL7_ROOT
    / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
)

print(
    "\nTraining history:"
)

print(
    HISTORY_FILE
)

print(
    "\nTraining summary:"
)

print(
    SUMMARY_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Do not evaluate the frozen test set until "
    "the training output has been reviewed."
)

NOTEBOOK 13 — CELL 7
FINAL GENOMIC TRANSFORMER TRAINING

PATH GATE
--------------------------------------------------------------------------------
Sequence dataset    : PASS
Vocabulary          : PASS
Architecture        : PASS

FROZEN DATASET
--------------------------------------------------------------------------------
Full: 1976
Train: 1185
Validation: 395
Test: 396
Frozen cohort: PASS

TRAINING TARGET DISTRIBUTION
--------------------------------------------------------------------------------
Train DS: 422
Train DR: 763
Validation DS: 141
Validation DR: 254

DEVICE
--------------------------------------------------------------------------------
Device: cpu

CLASS WEIGHTS
--------------------------------------------------------------------------------
DS weight: 1.4040284156799316
DR weight: 0.7765399813652039


STARTING FIXED-EPOCH TRAINING
Epochs: 40
Early stopping: DISABLED
Checkpoint selection: VALIDATION ROC-AUC
Test evaluation: NOT PERFORMED
Epoch 01/40 | Train Loss 0.3496

In [10]:
# =============================================================================
# NOTEBOOK 13 — CELL 8
# FINAL GENOMIC TRANSFORMER
# FROZEN TEST-SET EVALUATION
# =============================================================================

from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)


print("=" * 80)
print("NOTEBOOK 13 — CELL 8")
print("FINAL GENOMIC TRANSFORMER — FROZEN TEST EVALUATION")
print("=" * 80)


# =============================================================================
# 1. REPRODUCIBILITY
# =============================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =============================================================================
# 2. PATHS
# =============================================================================

PROJECT_ROOT = Path(
    r"C:\TBP\Metadata"
)

FINAL_ROOT = (
    PROJECT_ROOT
    / "Step_3B_8_CXR_CoAtNet_Preparation"
    / "QC"
    / "Clean_PSPNet_CXR_Cohort"
    / "FINAL_FROZEN_CLEAN_COHORT"
)

CELL5_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell5_Training_Only_Vocabulary"
)

CELL7_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell7_Transformer_Training"
)

CELL8_ROOT = (
    FINAL_ROOT
    / "Genomic_Preparation_13"
    / "Cell8_Frozen_Test_Evaluation"
)

CELL8_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


SEQUENCE_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_FINAL_1976_Genomic_Sequences.csv"
)

VOCAB_FILE = (
    CELL5_ROOT
    / "Notebook13_Cell5_TRAIN_ONLY_Vocabulary.json"
)

CHECKPOINT_FILE = (
    CELL7_ROOT
    / "Notebook13_Cell7_Best_Validation_ROC_AUC.pth"
)

TRAINING_SUMMARY_FILE = (
    CELL7_ROOT
    / "Notebook13_Cell7_Training_Summary.json"
)


# =============================================================================
# 3. PATH GATE
# =============================================================================

print("\nPATH GATE")
print("-" * 80)

required_paths = {

    "Sequence dataset":
        SEQUENCE_FILE,

    "Training-only vocabulary":
        VOCAB_FILE,

    "Best validation checkpoint":
        CHECKPOINT_FILE,

    "Training summary":
        TRAINING_SUMMARY_FILE,

}

for name, path in required_paths.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(
        f"{name:<30}: PASS"
    )


# =============================================================================
# 4. LOAD DATA
# =============================================================================

sequence_df = pd.read_csv(
    SEQUENCE_FILE,
    low_memory=False
)

with open(
    VOCAB_FILE,
    "r",
    encoding="utf-8"
) as f:

    vocab_data = json.load(f)

with open(
    TRAINING_SUMMARY_FILE,
    "r",
    encoding="utf-8"
) as f:

    training_summary = json.load(f)


# =============================================================================
# 5. CHECK BEST CHECKPOINT
# =============================================================================

expected_best_epoch = int(
    training_summary[
        "best_epoch"
    ]
)

expected_best_val_auc = float(
    training_summary[
        "best_validation_roc_auc"
    ]
)

print("\nCHECKPOINT INFORMATION")
print("-" * 80)

print(
    "Selected checkpoint epoch:",
    expected_best_epoch
)

print(
    "Validation ROC-AUC:",
    f"{expected_best_val_auc:.6f}"
)

if expected_best_epoch != 11:

    raise RuntimeError(
        "Expected best checkpoint at epoch 11 "
        "based on Cell 7 output."
    )

if not (
    np.isclose(
        expected_best_val_auc,
        0.975764,
        atol=1e-5
    )
):

    print(
        "WARNING: stored summary value differs "
        "slightly from displayed Cell 7 output."
    )


# =============================================================================
# 6. FINAL COHORT GATE
# =============================================================================

if len(sequence_df) != 1976:

    raise RuntimeError(
        "Expected exactly 1976 conditions."
    )

if sequence_df[
    "condition_id"
].nunique() != 1976:

    raise RuntimeError(
        "Condition IDs are not unique."
    )


expected_split_counts = {

    "train":
        1185,

    "validation":
        395,

    "test":
        396,

}

actual_split_counts = (
    sequence_df[
        "split"
    ]
    .value_counts()
    .to_dict()
)

for split_name, expected in (
    expected_split_counts.items()
):

    actual = actual_split_counts.get(
        split_name,
        0
    )

    if actual != expected:

        raise RuntimeError(
            f"{split_name}: expected "
            f"{expected}, found {actual}"
        )


print("\nFINAL COHORT")
print("-" * 80)

print(
    "Full:",
    len(sequence_df)
)

print(
    "Train:",
    actual_split_counts["train"]
)

print(
    "Validation:",
    actual_split_counts["validation"]
)

print(
    "Test:",
    actual_split_counts["test"]
)

print(
    "Cohort gate: PASS"
)


# =============================================================================
# 7. CREATE ONLY THE FROZEN TEST DATASET
# =============================================================================

test_df = sequence_df[
    sequence_df["split"] == "test"
].copy()


if len(test_df) != 396:

    raise RuntimeError(
        "Frozen test set must contain exactly 396 conditions."
    )


# =============================================================================
# 8. TEST TARGET GATE
# =============================================================================

test_ds_count = int(
    (
        test_df[
            "target_binary"
        ] == 0
    ).sum()
)

test_dr_count = int(
    (
        test_df[
            "target_binary"
        ] == 1
    ).sum()
)

print("\nFROZEN TEST TARGET")
print("-" * 80)

print(
    "DS:",
    test_ds_count
)

print(
    "DR:",
    test_dr_count
)

if test_ds_count != 141:

    raise RuntimeError(
        "Frozen test DS count changed."
    )

if test_dr_count != 255:

    raise RuntimeError(
        "Frozen test DR count changed."
    )

print(
    "Test target distribution: PASS"
)


# =============================================================================
# 9. PARSE STORED SEQUENCES
# =============================================================================

def parse_array(value):

    value = str(
        value
    ).strip()

    value = (
        value
        .replace("[", "")
        .replace("]", "")
    )

    if not value:

        return []

    return [
        int(x.strip())
        for x in value.split(",")
        if x.strip()
    ]


test_df[
    "input_ids_parsed"
] = test_df[
    "input_ids"
].apply(
    parse_array
)

test_df[
    "attention_mask_parsed"
] = test_df[
    "attention_mask"
].apply(
    parse_array
)


MAX_SEQ_LEN = int(
    vocab_data[
        "max_seq_len"
    ]
)

VOCAB_SIZE = int(
    vocab_data[
        "vocab_size"
    ]
)

PAD_ID = int(
    vocab_data[
        "pad_id"
    ]
)


# =============================================================================
# 10. TEST SEQUENCE VALIDATION
# =============================================================================

for ids in test_df[
    "input_ids_parsed"
]:

    if len(ids) != MAX_SEQ_LEN:

        raise RuntimeError(
            "Test sequence has incorrect length."
        )

for mask in test_df[
    "attention_mask_parsed"
]:

    if len(mask) != MAX_SEQ_LEN:

        raise RuntimeError(
            "Test attention mask has incorrect length."
        )

all_test_ids = np.concatenate(
    [
        np.asarray(
            ids,
            dtype=np.int64
        )
        for ids in test_df[
            "input_ids_parsed"
        ]
    ]
)

if all_test_ids.min() < 0:

    raise RuntimeError(
        "Negative token ID in test set."
    )

if all_test_ids.max() >= VOCAB_SIZE:

    raise RuntimeError(
        "Test token ID exceeds vocabulary size."
    )


# =============================================================================
# 11. TEST DATASET
# =============================================================================

class GenomicTestDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
        )

        self.input_ids = np.asarray(
            self.df[
                "input_ids_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.attention_masks = np.asarray(
            self.df[
                "attention_mask_parsed"
            ].tolist(),
            dtype=np.int64
        )

        self.targets = (
            self.df[
                "target_binary"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        self.condition_ids = (
            self.df[
                "condition_id"
            ]
            .astype(str)
            .to_numpy()
        )

    def __len__(
        self
    ):

        return len(
            self.targets
        )

    def __getitem__(
        self,
        index
    ):

        return {

            "input_ids":
                torch.tensor(
                    self.input_ids[index],
                    dtype=torch.long
                ),

            "attention_mask":
                torch.tensor(
                    self.attention_masks[index],
                    dtype=torch.long
                ),

            "target":
                torch.tensor(
                    self.targets[index],
                    dtype=torch.long
                ),

            "condition_id":
                self.condition_ids[index],

        }


test_dataset = GenomicTestDataset(
    test_df
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)


# =============================================================================
# 12. MODEL
# =============================================================================

class GenomicTransformer(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        ff_dim=256,
        dropout=0.20,
        max_seq_len=74,
        num_classes=2,
        pad_id=0
    ):

        super().__init__()

        self.embed_dim = embed_dim

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_id
        )

        self.position_embedding = nn.Embedding(
            max_seq_len,
            embed_dim
        )

        encoder_layer = (
            nn.TransformerEncoderLayer(
                d_model=embed_dim,
                nhead=num_heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=False
            )
        )

        self.encoder = (
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                embed_dim
            ),
            nn.Linear(
                embed_dim,
                num_classes
            )
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        return_embedding=False
    ):

        batch_size, seq_len = (
            input_ids.shape
        )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        positions = (
            positions
            .unsqueeze(0)
            .expand(
                batch_size,
                seq_len
            )
        )

        x = (
            self.token_embedding(
                input_ids
            )
            +
            self.position_embedding(
                positions
            )
        )

        x = self.dropout(
            x
        )

        padding_mask = (
            attention_mask == 0
        )

        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        cls_embedding = x[:, 0, :]

        cls_embedding = self.dropout(
            cls_embedding
        )

        logits = self.classifier(
            cls_embedding
        )

        if return_embedding:

            return (
                logits,
                cls_embedding
            )

        return logits


# =============================================================================
# 13. DEVICE
# =============================================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDEVICE")
print("-" * 80)

print(
    "Device:",
    device
)


# =============================================================================
# 14. LOAD BEST VALIDATION CHECKPOINT
# =============================================================================

model = GenomicTransformer(
    vocab_size=VOCAB_SIZE,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    ff_dim=256,
    dropout=0.20,
    max_seq_len=MAX_SEQ_LEN,
    num_classes=2,
    pad_id=PAD_ID
).to(device)


checkpoint = torch.load(
    CHECKPOINT_FILE,
    map_location=device,
    weights_only=False
)

checkpoint_epoch = int(
    checkpoint[
        "epoch"
    ]
)

checkpoint_auc = float(
    checkpoint[
        "validation_roc_auc"
    ]
)

print("\nCHECKPOINT LOAD")
print("-" * 80)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Checkpoint validation ROC-AUC:",
    f"{checkpoint_auc:.6f}"
)

if checkpoint_epoch != expected_best_epoch:

    raise RuntimeError(
        "Checkpoint epoch does not match "
        "the best validation epoch."
    )

if not np.isclose(
    checkpoint_auc,
    expected_best_val_auc,
    atol=1e-6
):

    raise RuntimeError(
        "Checkpoint validation ROC-AUC does not "
        "match the recorded best validation result."
    )


model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

model.eval()

print(
    "Best validation checkpoint loaded: PASS"
)


# =============================================================================
# 15. FROZEN TEST INFERENCE
# =============================================================================

print("\n")
print("=" * 80)
print("FROZEN TEST INFERENCE")
print("=" * 80)

all_targets = []
all_probabilities = []
all_predictions = []
all_condition_ids = []
all_embeddings = []


with torch.no_grad():

    for batch in test_loader:

        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        targets = batch[
            "target"
        ].to(device)

        condition_ids = batch[
            "condition_id"
        ]

        logits, embeddings = model(
            input_ids,
            attention_mask,
            return_embedding=True
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).long()

        all_targets.extend(
            targets.cpu()
            .numpy()
            .tolist()
        )

        all_probabilities.extend(
            probabilities.cpu()
            .numpy()
            .tolist()
        )

        all_predictions.extend(
            predictions.cpu()
            .numpy()
            .tolist()
        )

        all_condition_ids.extend(
            list(
                condition_ids
            )
        )

        all_embeddings.append(
            embeddings.cpu()
            .numpy()
        )


# =============================================================================
# 16. PREDICTION COUNT GATE
# =============================================================================

if len(all_targets) != 396:

    raise RuntimeError(
        f"Expected 396 test predictions, "
        f"received {len(all_targets)}."
    )

if len(
    set(
        all_condition_ids
    )
) != 396:

    raise RuntimeError(
        "Test condition IDs are not unique."
    )

print(
    "Test predictions:",
    len(all_targets)
)

print(
    "Prediction count: PASS"
)


# =============================================================================
# 17. METRICS
# =============================================================================

y_true = np.asarray(
    all_targets,
    dtype=np.int64
)

y_probability = np.asarray(
    all_probabilities,
    dtype=np.float64
)

y_pred = np.asarray(
    all_predictions,
    dtype=np.int64
)


accuracy = accuracy_score(
    y_true,
    y_pred
)

balanced_accuracy = (
    balanced_accuracy_score(
        y_true,
        y_pred
    )
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

sensitivity = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

roc_auc = roc_auc_score(
    y_true,
    y_probability
)

pr_auc = average_precision_score(
    y_true,
    y_probability
)

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    cm.ravel()
)

specificity = (
    tn
    /
    (tn + fp)
    if (tn + fp) > 0
    else 0.0
)


# =============================================================================
# 18. TEST RESULTS
# =============================================================================

print("\n")
print("=" * 80)
print("FINAL FROZEN TEST RESULTS")
print("=" * 80)

print(
    f"Accuracy           : {accuracy:.6f}"
)

print(
    f"Balanced Accuracy  : {balanced_accuracy:.6f}"
)

print(
    f"Precision           : {precision:.6f}"
)

print(
    f"Sensitivity / Recall: {sensitivity:.6f}"
)

print(
    f"Specificity         : {specificity:.6f}"
)

print(
    f"F1 Score            : {f1:.6f}"
)

print(
    f"ROC-AUC             : {roc_auc:.6f}"
)

print(
    f"PR-AUC              : {pr_auc:.6f}"
)


# =============================================================================
# 19. CONFUSION MATRIX
# =============================================================================

print("\nCONFUSION MATRIX")
print("-" * 80)

print(
    "                 Predicted"
)

print(
    "                 DS     DR"
)

print(
    f"Actual DS       {tn:4d}   {fp:4d}"
)

print(
    f"Actual DR       {fn:4d}   {tp:4d}"
)


# =============================================================================
# 20. CLASSIFICATION REPORT
# =============================================================================

print("\nCLASSIFICATION REPORT")
print("-" * 80)

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1],
        target_names=[
            "DS-TB",
            "DR-TB"
        ],
        digits=4,
        zero_division=0
    )
)


# =============================================================================
# 21. PREDICTED DISTRIBUTION
# =============================================================================

print("PREDICTED DISTRIBUTION")
print("-" * 80)

print(
    "Predicted DS:",
    int(
        (
            y_pred == 0
        ).sum()
    )
)

print(
    "Predicted DR:",
    int(
        (
            y_pred == 1
        ).sum()
    )
)

print(
    "Actual DS:",
    int(
        (
            y_true == 0
        ).sum()
    )
)

print(
    "Actual DR:",
    int(
        (
            y_true == 1
        ).sum()
    )
)


# =============================================================================
# 22. EMBEDDING SHAPE
# =============================================================================

test_embeddings = np.concatenate(
    all_embeddings,
    axis=0
)

print("\nTEST EMBEDDING")
print("-" * 80)

print(
    "Shape:",
    test_embeddings.shape
)

if test_embeddings.shape != (
    396,
    128
):

    raise RuntimeError(
        "Unexpected test embedding shape."
    )


# =============================================================================
# 23. SAVE TEST PREDICTIONS
# =============================================================================

prediction_df = pd.DataFrame(
    {
        "condition_id":
            all_condition_ids,

        "target_binary":
            y_true,

        "predicted_probability_DR":
            y_probability,

        "predicted_class":
            y_pred,

    }
)

PREDICTIONS_FILE = (
    CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Frozen_Test_Predictions.csv"
)

prediction_df.to_csv(
    PREDICTIONS_FILE,
    index=False
)


# =============================================================================
# 24. SAVE TEST EMBEDDINGS
# =============================================================================

TEST_EMBEDDINGS_FILE = (
    CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Genomic_Embeddings_128D.npy"
)

TEST_EMBEDDING_IDS_FILE = (
    CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Condition_IDs.npy"
)

np.save(
    TEST_EMBEDDINGS_FILE,
    test_embeddings
)

np.save(
    TEST_EMBEDDING_IDS_FILE,
    np.asarray(
        all_condition_ids,
        dtype=str
    )
)


# =============================================================================
# 25. SAVE METRICS
# =============================================================================

metrics = {

    "model":
        "Final Genomic Transformer",

    "checkpoint_epoch":
        checkpoint_epoch,

    "validation_roc_auc_at_checkpoint":
        checkpoint_auc,

    "test_conditions":
        396,

    "test_ds":
        141,

    "test_dr":
        255,

    "accuracy":
        float(accuracy),

    "balanced_accuracy":
        float(balanced_accuracy),

    "precision":
        float(precision),

    "sensitivity":
        float(sensitivity),

    "specificity":
        float(specificity),

    "f1":
        float(f1),

    "roc_auc":
        float(roc_auc),

    "pr_auc":
        float(pr_auc),

    "true_negative":
        int(tn),

    "false_positive":
        int(fp),

    "false_negative":
        int(fn),

    "true_positive":
        int(tp),

    "classification_threshold":
        0.5,

    "threshold_tuned_on_test":
        False,

    "checkpoint_selected_using_test":
        False,

    "model_retrained_after_test":
        False,

    "test_evaluated":
        True,

    "embedding_dimension":
        128,

    "status":
        "FROZEN_TEST_EVALUATION_COMPLETE"

}


METRICS_FILE = (
    CELL8_ROOT
    / "Notebook13_Cell8_FINAL_Test_Metrics.json"
)

with open(
    METRICS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metrics,
        f,
        indent=4
    )


# =============================================================================
# 26. FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)
print("CELL 8 STATUS: PASS")
print("=" * 80)

print(
    "Checkpoint epoch:",
    checkpoint_epoch
)

print(
    "Test conditions:",
    396
)

print(
    "Test threshold:",
    0.5
)

print(
    "Threshold tuned using test:",
    "NO"
)

print(
    "Checkpoint selected using test:",
    "NO"
)

print(
    "\nFinal test ROC-AUC:",
    f"{roc_auc:.6f}"
)

print(
    "Final test accuracy:",
    f"{accuracy:.6f}"
)

print(
    "Final test F1:",
    f"{f1:.6f}"
)

print(
    "\nPredictions:"
)

print(
    PREDICTIONS_FILE
)

print(
    "\nTest embeddings:"
)

print(
    TEST_EMBEDDINGS_FILE
)

print(
    "\nMetrics:"
)

print(
    METRICS_FILE
)

print(
    "\nSTOP HERE."
)

print(
    "Do not retrain or tune the genomic model based "
    "on these test results."
)

NOTEBOOK 13 — CELL 8
FINAL GENOMIC TRANSFORMER — FROZEN TEST EVALUATION

PATH GATE
--------------------------------------------------------------------------------
Sequence dataset              : PASS
Training-only vocabulary      : PASS
Best validation checkpoint    : PASS
Training summary              : PASS

CHECKPOINT INFORMATION
--------------------------------------------------------------------------------
Selected checkpoint epoch: 11
Validation ROC-AUC: 0.975764

FINAL COHORT
--------------------------------------------------------------------------------
Full: 1976
Train: 1185
Validation: 395
Test: 396
Cohort gate: PASS

FROZEN TEST TARGET
--------------------------------------------------------------------------------
DS: 141
DR: 255
Test target distribution: PASS

DEVICE
--------------------------------------------------------------------------------
Device: cpu

CHECKPOINT LOAD
--------------------------------------------------------------------------------
Checkpoint epoc